# DS102 V52 — CSV thử nghiệm, không pretrained
Đã tạo CSV 3974dòng,11nhãn khác file tốt nhất từng đạtKaggle0.77436. **Chưa biết điểm Kaggle của V52.**
Cấu hình cố định: chỉ đổi nhãn ngoài lớp fake review khi cả hai nhánh mô tả sản phẩm và SVD có trọng số đều đồng ý. Giữ nguyên toàn bộ quyết định lớp1 của bản nền.
OOF0.774945→0.776409;dev0.726349→0.728946. **Ngưỡng CV0.002 và bộ điều kiện dev bổ sung không đạt**; các báo cáo bị loại vẫn được giữ. Đây là bản xuất để kiểm chứng thử nghiệm, không phải mô hình đã vượt mọi gate.
Bootstrap OOF95%[0.000162;0.002795];dev95%[-0.001891;0.008294]. Dev vẫn có thể giảm theo khoảng bất định; OOF/dev đã dùng nhiều lần và bootstrap không loại bỏ thiên lệch chọn mô hình. Không bảo đảm tăng điểm leaderboard.
Notebook fit lại hai nhánh từ đúng train trên hostedColabCPU,không pretrained hoặc dữ liệu ngoài. Dev/test không được dùng fit. Dùng **DS102_v43_inputs.zip** đã có (19MB), chứa4CSV và dự đoán nền V11 tự train trước đây.
TảiZIP bằng bảng Tệp bên trái rồi chọn Runtime→Run all. Notebook nhậnZIP đã tải; nếu chưa có sẽ hỏi tải lên. Giữtab mở. Không tự submitKaggle.
Gửi feedback_v52.zip và điểm Kaggle nếu bạn nộp thửCSV. Nguồn đầy đủ được nhúng; kiểm tra source/cú pháp đã chạy. Chưa chạy lại notebook sạch trọn vẹn; notebook fit mô hình cuối và dùng lại bằng chứngCV đã lưu,không chạy lại toàn bộ nghiên cứuCV.


## 1. Chuẩn bị môi trường Colab

In [ ]:
# Chỉ chạy trên máy chủ Colab. Không chọn local runtime.
import os,platform,sys,subprocess
from pathlib import Path
assert platform.system()=='Linux' and os.environ.get('COLAB_RELEASE_TAG'), 'Hãy chạy trên hosted Google Colab.'
ROOT43=Path('/content/ds102_v43');ROOT43.mkdir(exist_ok=True)
PY43=ROOT43/'venv/bin/python'
subprocess.check_call([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(ROOT43/'venv')])
packages=['numpy==2.4.6', 'scipy==1.16.3', 'pandas==2.2.3', 'scikit-learn==1.7.2', 'catboost==1.2.10']
subprocess.check_call([sys.executable,'-m','pip','--python',str(PY43),'install','--quiet','--disable-pip-version-check',*packages])
print('Đã chuẩn bị môi trường Colab riêng; không cần khởi động lại kernel.')


## 2. Tải gói đầu vào

In [ ]:
import hashlib,json,zipfile
from google.colab import files
PACK43=ROOT43/'DS102_v43_inputs.zip'
EXPECTED_PACK_SHA='19c942cbf6e615591dec5cf1e248f3130ee65793edcdd21f1a38df17e92d2a5a'
# Hỗ trợ ZIP đã tải bằng bảng Tệp bên trái khi tiện ích trong ô bị chặn.
import shutil
if not PACK43.exists():
    for existing51 in Path('/content').glob('DS102_v43_inputs*.zip'):
        if hashlib.sha256(existing51.read_bytes()).hexdigest()==EXPECTED_PACK_SHA:
            shutil.copy2(existing51,PACK43)
            break
if not PACK43.exists():
    print('Chọn file DS102_v43_inputs.zip đã tải về máy.')
    uploaded43=files.upload()
    chosen43=next((b for n,b in uploaded43.items() if hashlib.sha256(b).hexdigest()==EXPECTED_PACK_SHA),None)
    assert chosen43 is not None,'Cần chọn đúng gói DS102_v43_inputs.zip đi kèm notebook này.'
    PACK43.write_bytes(chosen43)
    del uploaded43,chosen43
assert hashlib.sha256(PACK43.read_bytes()).hexdigest()==EXPECTED_PACK_SHA,'Gói đầu vào khác phiên bản notebook này.'
INPUTS43=ROOT43/'inputs';INPUTS43.mkdir(exist_ok=True)
with zipfile.ZipFile(PACK43) as z:
    for info in z.infolist():
        target43=(INPUTS43/info.filename).resolve()
        assert target43.is_relative_to(INPUTS43.resolve()),'Đường dẫn ZIP không hợp lệ'
    assert sum(i.file_size for i in z.infolist())<200_000_000
    z.extractall(INPUTS43)
manifest43=json.loads((INPUTS43/'inputs_manifest.json').read_text())
for n,h in manifest43['files'].items():
    assert hashlib.sha256((INPUTS43/n).read_bytes()).hexdigest()==h,n
print('Đã kiểm tra dữ liệu và các dự đoán nền từ những lượt train trước.')


## 3. Ghi mã nguồn

In [ ]:
import json
from pathlib import Path
ROOT51=Path('/content/ds102_v51');ROOT51.mkdir(exist_ok=True)
SRC43=ROOT51/'source';SRC43.mkdir(exist_ok=True)
modules51=json.loads('{"train_submit.py": "\\"\\"\\"CPU baseline cho DS102: TF-IDF + Linear SVM, chọn theo F1-macro.\\n\\nChỉ dùng comment và nhãn train. Dev được đánh giá sau khi chọn tham số\\nbằng cross-validation trên train; không gộp dev vào train.\\n\\"\\"\\"\\n\\nimport argparse\\nimport html\\nimport json\\nimport platform\\nimport re\\nimport unicodedata\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom scipy import sparse\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import LinearSVC\\n\\n\\nSEED = 102\\nLABELS = [0, 1, 2, 3]\\nFILES = {name: f\\"ds102_{name}.csv\\" for name in [\\"train\\", \\"dev\\", \\"test\\", \\"samplesubmission\\"]}\\n\\n\\ndef normalize_text(value):\\n    \\"\\"\\"Giữ dấu, emoji, dấu câu và ký tự lặp vì có thể giúp nhận diện spam.\\"\\"\\"\\n    text = html.unescape(unicodedata.normalize(\\"NFC\\", str(value)))\\n    text = text.replace(\\"\\\\\\\\r\\\\\\\\n\\", \\"\\\\n\\").replace(\\"\\\\\\\\n\\", \\"\\\\n\\").replace(\\"\\\\\\\\r\\", \\"\\\\n\\")\\n    return re.sub(r\\"\\\\s+\\", \\" \\", text).strip().lower()\\n\\n\\nclass ReviewStats(BaseEstimator, TransformerMixin):\\n    \\"\\"\\"Đặc trưng độ dài, lặp ký tự và tỷ lệ ký tự; không có từ khóa nhãn.\\"\\"\\"\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def transform(self, X):\\n        rows = []\\n        for text in X:\\n            chars = len(text)\\n            words = text.split()\\n            nwords = len(words)\\n            denom = max(chars, 1)\\n            word_denom = max(nwords, 1)\\n            letters = [c for c in text if c.isalpha()]\\n            longest_run = max((len(m.group()) for m in re.finditer(r\\"(.)\\\\1*\\", text)), default=0)\\n            rows.append([\\n                np.log1p(chars), np.log1p(nwords), chars / word_denom,\\n                len(set(words)) / word_denom, len(set(text)) / denom,\\n                longest_run / denom, np.log1p(longest_run),\\n                sum(c.isdigit() for c in text) / denom,\\n                len(letters) / denom,\\n                sum(not c.isalnum() and not c.isspace() for c in text) / denom,\\n                sum(ord(c) > 0xFFFF for c in text) / denom,\\n                text.count(\\"!\\") / denom, text.count(\\"?\\") / denom,\\n                float(bool(re.search(r\\"https?://|www\\\\.\\", text))),\\n            ])\\n        return sparse.csr_matrix(np.asarray(rows, dtype=np.float32))\\n\\n\\ndef make_features():\\n    return FeatureUnion([\\n        (\\"word\\", TfidfVectorizer(\\n            ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=90000,\\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\\n            token_pattern=r\\"(?u)\\\\b\\\\w+\\\\b\\",\\n        )),\\n        (\\"char\\", TfidfVectorizer(\\n            analyzer=\\"char\\", ngram_range=(2, 5), min_df=3, max_features=160000,\\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\\n        )),\\n        (\\"stats\\", Pipeline([\\n            (\\"extract\\", ReviewStats()), (\\"scale\\", StandardScaler(with_mean=False)),\\n        ])),\\n    ], transformer_weights={\\"word\\": 1.0, \\"char\\": 1.0, \\"stats\\": 0.1})\\n\\n\\ndef make_model(config, y):\\n    weight = config[\\"weight\\"]\\n    if weight == \\"sqrt_balanced\\":\\n        counts = pd.Series(y).value_counts()\\n        weight = {int(k): float(np.sqrt(len(y) / (len(counts) * v))) for k, v in counts.items()}\\n    elif weight == \\"none\\":\\n        weight = None\\n    return LinearSVC(C=config[\\"C\\"], class_weight=weight, dual=True,\\n                     max_iter=10000, tol=1e-4, random_state=SEED)\\n\\n\\ndef find_data_dir(supplied=None):\\n    if supplied:\\n        path = Path(supplied).expanduser().resolve()\\n        if all((path / f).is_file() for f in FILES.values()):\\n            return path\\n        raise FileNotFoundError(f\\"Thiếu một trong 4 file CSV tại {path}\\")\\n    candidates = [Path.cwd(), Path.cwd() / \\"data\\", Path.cwd() / \\"dataset\\"]\\n    kaggle = Path(\\"/kaggle/input\\")\\n    if kaggle.exists():\\n        candidates += sorted({p.parent for p in kaggle.rglob(FILES[\\"train\\"])})\\n    matches = [p for p in candidates if all((p / f).is_file() for f in FILES.values())]\\n    if len(matches) == 1:\\n        return matches[0].resolve()\\n    raise FileNotFoundError(\\"Hãy truyền --data-dir đến thư mục chứa đúng 4 CSV; trên Kaggle dùng Add Input trước.\\")\\n\\n\\ndef load_data(data_dir):\\n    data = {name: pd.read_csv(data_dir / file, keep_default_na=False) for name, file in FILES.items()}\\n    for name in [\\"train\\", \\"dev\\", \\"test\\"]:\\n        required = {\\"comment\\", \\"spam_label\\"} if name != \\"test\\" else {\\"comment\\", \\"id\\"}\\n        if not required.issubset(data[name].columns):\\n            raise ValueError(f\\"{name}: thiếu cột {required - set(data[name].columns)}\\")\\n        if data[name].empty:\\n            raise ValueError(f\\"{name} rỗng\\")\\n    for name in [\\"train\\", \\"dev\\"]:\\n        values = pd.to_numeric(data[name][\\"spam_label\\"], errors=\\"raise\\")\\n        if not values.isin(LABELS).all():\\n            raise ValueError(f\\"{name}: nhãn phải thuộc {LABELS}\\")\\n        data[name][\\"spam_label\\"] = values.astype(int)\\n    if set(data[\\"train\\"][\\"spam_label\\"]) != set(LABELS):\\n        raise ValueError(\\"Train phải có đủ 4 nhãn\\")\\n    sample = data[\\"samplesubmission\\"]\\n    if list(sample.columns) != [\\"ID\\", \\"spam_label\\"]:\\n        raise ValueError(\\"Sample phải có hai cột ID,spam_label\\")\\n    test_ids, sample_ids = data[\\"test\\"][\\"id\\"].astype(str), sample[\\"ID\\"].astype(str)\\n    if not test_ids.is_unique or not sample_ids.is_unique or set(test_ids) != set(sample_ids):\\n        raise ValueError(\\"ID trong test và sample phải duy nhất và trùng tập hợp\\")\\n    return data\\n\\n\\ndef metrics(y, pred):\\n    return {\\n        \\"macro_f1\\": float(f1_score(y, pred, labels=LABELS, average=\\"macro\\", zero_division=0)),\\n        \\"accuracy\\": float(accuracy_score(y, pred)),\\n        \\"classification_report\\": classification_report(y, pred, labels=LABELS, output_dict=True, zero_division=0),\\n        \\"confusion_matrix\\": confusion_matrix(y, pred, labels=LABELS).tolist(),\\n    }\\n\\n\\ndef main(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\"--data-dir\\", default=None)\\n    parser.add_argument(\\"--output-dir\\", default=\\"output\\")\\n    parser.add_argument(\\"--folds\\", type=int, default=3)\\n    args = parser.parse_args(argv)\\n    if args.folds < 2:\\n        parser.error(\\"--folds phải >= 2\\")\\n    data_dir = find_data_dir(args.data_dir)\\n    output = Path(args.output_dir).expanduser().resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    data = load_data(data_dir)\\n    train, dev, test = [data[n] for n in [\\"train\\", \\"dev\\", \\"test\\"]]\\n    texts = train[\\"comment\\"].map(normalize_text).to_numpy()\\n    dev_texts = dev[\\"comment\\"].map(normalize_text).to_numpy()\\n    test_texts = test[\\"comment\\"].map(normalize_text).to_numpy()\\n    y = train[\\"spam_label\\"].to_numpy()\\n    if train.groupby(train[\\"comment\\"].map(normalize_text))[\\"spam_label\\"].first().value_counts().min() < args.folds:\\n        raise ValueError(\\"Không đủ nhóm cho số folds đã chọn\\")\\n    print(f\\"Train={len(train)}, dev={len(dev)}, test={len(test)}\\", flush=True)\\n    print(f\\"Nhãn train: {train.spam_label.value_counts().sort_index().to_dict()}\\", flush=True)\\n    configs = [{\\"C\\": c, \\"weight\\": weight} for weight in [\\"none\\", \\"sqrt_balanced\\", \\"balanced\\"] for c in [1.0, 4.0]]\\n    oof = np.full((len(configs), len(y)), -1, dtype=np.int64)\\n    fold_scores = [[] for _ in configs]\\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\\n    for fold, (fit_idx, val_idx) in enumerate(splitter.split(texts, y, groups=texts), 1):\\n        print(f\\"Fold {fold}/{args.folds}: fit TF-IDF chỉ trên train của fold...\\", flush=True)\\n        features = make_features()\\n        X_fit = features.fit_transform(texts[fit_idx])\\n        X_val = features.transform(texts[val_idx])\\n        for i, config in enumerate(configs):\\n            model = make_model(config, y[fit_idx])\\n            model.fit(X_fit, y[fit_idx])\\n            pred = model.predict(X_val)\\n            oof[i, val_idx] = pred\\n            score = f1_score(y[val_idx], pred, labels=LABELS, average=\\"macro\\", zero_division=0)\\n            fold_scores[i].append(float(score))\\n            print(f\\"  {config}: F1-macro={score:.5f}\\", flush=True)\\n        del X_fit, X_val, features\\n    cv_scores = [float(f1_score(y, p, labels=LABELS, average=\\"macro\\", zero_division=0)) for p in oof]\\n    best_idx = int(np.argmax(cv_scores))\\n    best = configs[best_idx]\\n    print(f\\"Chọn bằng OOF train: {best}, F1-macro={cv_scores[best_idx]:.5f}\\", flush=True)\\n    print(\\"Huấn luyện cấu hình đã chọn trên toàn bộ train...\\", flush=True)\\n    features = make_features()\\n    X_train = features.fit_transform(texts)\\n    model = make_model(best, y)\\n    model.fit(X_train, y)\\n    dev_pred = model.predict(features.transform(dev_texts))\\n    dev_y = dev[\\"spam_label\\"].to_numpy()\\n    report = metrics(dev_y, dev_pred)\\n    unique_dev = ~pd.Series(dev_texts).isin(set(texts)).to_numpy()\\n    print(f\\"Dev F1-macro={report[\'macro_f1\']:.5f}, accuracy={report[\'accuracy\']:.5f}\\", flush=True)\\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\\n    print(f\\"Dev có {int((~unique_dev).sum())} comment chuẩn hóa xuất hiện trong train.\\", flush=True)\\n    test_pred = model.predict(features.transform(test_texts)).astype(int)\\n    by_id = pd.Series(test_pred, index=test[\\"id\\"].astype(str))\\n    submission = data[\\"samplesubmission\\"].copy()\\n    submission[\\"spam_label\\"] = submission[\\"ID\\"].astype(str).map(by_id)\\n    if submission[\\"spam_label\\"].isna().any() or not submission[\\"spam_label\\"].isin(LABELS).all():\\n        raise ValueError(\\"Submission có nhãn thiếu hoặc không hợp lệ\\")\\n    submission[\\"spam_label\\"] = submission[\\"spam_label\\"].astype(int)\\n    submission_path = output / \\"submission.csv\\"\\n    submission.to_csv(submission_path, index=False)\\n    saved = pd.read_csv(submission_path)\\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\\n    if len(saved) != len(test) or not saved[\\"ID\\"].equals(data[\\"samplesubmission\\"][\\"ID\\"]):\\n        raise ValueError(\\"Sai số hàng hoặc thứ tự ID khi lưu\\")\\n    detail = pd.DataFrame({\\"comment\\": dev[\\"comment\\"], \\"true_label\\": dev_y, \\"predicted_label\\": dev_pred,\\n                           \\"comment_seen_in_train\\": ~unique_dev})\\n    detail.to_csv(output / \\"dev_predictions.csv\\", index=False)\\n    full_report = {\\n        \\"seed\\": SEED, \\"python\\": platform.python_version(), \\"sklearn\\": sklearn.__version__,\\n        \\"rows\\": {k: len(v) for k, v in data.items()}, \\"feature_columns\\": [\\"comment\\"],\\n        \\"selection\\": \\"StratifiedGroupKFold on train; normalized comments as groups; OOF macro F1\\",\\n        \\"fit_data\\": \\"train only; dev and test never used to fit TF-IDF, scaler or SVM\\",\\n        \\"candidates\\": [{**c, \\"fold_macro_f1\\": f, \\"oof_macro_f1\\": s}\\n                       for c, f, s in zip(configs, fold_scores, cv_scores)],\\n        \\"chosen_config\\": best, \\"dev\\": report,\\n        \\"dev_comments_seen_in_train\\": int((~unique_dev).sum()),\\n        \\"dev_without_comment_overlap\\": metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\\n        \\"test_prediction_counts\\": {int(k): int(v) for k, v in submission.spam_label.value_counts().items()},\\n        \\"submission_rows_verified\\": len(saved),\\n    }\\n    (output / \\"metrics.json\\").write_text(json.dumps(full_report, ensure_ascii=False, indent=2), encoding=\\"utf-8\\")\\n    print(f\\"Đã kiểm tra và lưu {len(saved)} dự đoán: {submission_path}\\", flush=True)\\n    return full_report\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "improved_features.py": "\\"\\"\\"Đặc trưng v2: văn bản review và mức trùng với thông tin sản phẩm.\\"\\"\\"\\nimport re\\nfrom collections import Counter\\nimport numpy as np\\nfrom scipy import sparse\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\\nfrom train_submit import normalize_text, ReviewStats\\n\\n\\ndef prepare(frame):\\n    frame = frame.copy()\\n    for col in [\'comment\', \'product_name\', \'description\', \'category\', \'categories\']:\\n        frame[col] = frame[col].fillna(\'\').astype(str).map(normalize_text)\\n    return frame\\n\\n\\ndef tokens(text):\\n    return re.findall(r\'(?u)\\\\b\\\\w+\\\\b\', text)\\n\\n\\ndef ngrams(words, n):\\n    return set(zip(*(words[i:] for i in range(n))))\\n\\n\\ndef dense_features(df):\\n    basic = ReviewStats().transform(df.comment).toarray()\\n    extra = []\\n    for row in df.itertuples(index=False):\\n        text = row.comment\\n        words = tokens(text)\\n        wl = [len(w) for w in words] or [0]\\n        wc = Counter(words)\\n        n = max(len(words), 1)\\n        feats = [np.log1p(max(wl)), np.mean(wl), np.std(wl),\\n                 sum(len(w) > 12 for w in words) / n,\\n                 max(wc.values(), default=0) / n,\\n                 sum(v * (v - 1) for v in wc.values()) / max(n * (n - 1), 1),\\n                 text.count(\'.\') / max(len(text), 1),\\n                 len(re.findall(r\'(.)\\\\1{3,}\', text)) / n,\\n                 np.log1p(len(text.split())),\\n                 float(row.rating) / 5,\\n                 np.log1p(max(float(row.num_sold), 0)),\\n                 np.log1p(max(float(row.num_review), 0))]\\n        for context in [row.product_name, row.description]:\\n            context_words = tokens(context)\\n            for size in [1, 2, 3, 4]:\\n                a, b = ngrams(words, size), ngrams(context_words, size)\\n                overlap = len(a & b)\\n                ratio = overlap / max(len(a), 1)\\n                feats += [ratio, ratio ** 2, ratio * np.log1p(len(words)), np.log1p(overlap)]\\n            feats += [float(len(text) > 10 and text in context), np.log1p(len(context))]\\n        extra.append(feats)\\n    return np.hstack([basic, np.asarray(extra, dtype=np.float32)])\\n\\n\\nclass FeaturesV2:\\n    def __init__(self):\\n        self.word = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=140000,\\n                                   sublinear_tf=True, lowercase=False, dtype=np.float32,\\n                                   token_pattern=r\'(?u)\\\\b\\\\w+\\\\b\')\\n        self.char = TfidfVectorizer(analyzer=\'char\', ngram_range=(2, 5), min_df=3,\\n                                   max_features=180000, sublinear_tf=True,\\n                                   lowercase=False, dtype=np.float32)\\n        self.product = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000,\\n                                      sublinear_tf=True, lowercase=False, dtype=np.float32)\\n        self.category = OneHotEncoder(handle_unknown=\'ignore\', dtype=np.float32)\\n        self.scale = StandardScaler()\\n\\n    def fit_transform(self, df):\\n        text = sparse.hstack([self.word.fit_transform(df.comment), self.char.fit_transform(df.comment)], format=\'csr\')\\n        meta = sparse.hstack([self.product.fit_transform(df.product_name) * .2,\\n                             self.category.fit_transform(df[[\'category\', \'rating\']].astype(str)) * .2,\\n                             sparse.csr_matrix(self.scale.fit_transform(dense_features(df))) * .15], format=\'csr\')\\n        return text, meta\\n\\n    def transform(self, df):\\n        text = sparse.hstack([self.word.transform(df.comment), self.char.transform(df.comment)], format=\'csr\')\\n        meta = sparse.hstack([self.product.transform(df.product_name) * .2,\\n                             self.category.transform(df[[\'category\', \'rating\']].astype(str)) * .2,\\n                             sparse.csr_matrix(self.scale.transform(dense_features(df))) * .15], format=\'csr\')\\n        return text, meta\\n\\n\\ndef nb_ratios(X, y):\\n    \\"\\"\\"Log-count ratio one-vs-rest, fit chỉ từ phần train của fold.\\"\\"\\"\\n    ratios = []\\n    for c in range(4):\\n        pos = np.asarray(X[y == c].sum(axis=0)).ravel() + 1\\n        neg = np.asarray(X[y != c].sum(axis=0)).ravel() + 1\\n        ratios.append(np.log((pos / pos.sum()) / (neg / neg.sum())))\\n    return ratios\\n", "select_v2.py": "\\"\\"\\"Chọn mô hình/ensemble và bias bằng OOF train, không xem nhãn dev.\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\nimport numpy as np\\n\\n\\ndef macro(y, pred):\\n    cm = np.bincount(4 * y + pred, minlength=16).reshape(4, 4)\\n    den = cm.sum(axis=0) + cm.sum(axis=1)\\n    return float(np.mean(np.divide(2 * np.diag(cm), den, out=np.zeros(4), where=den > 0)))\\n\\n\\ndef calibrate(scores, y):\\n    # 3 offsets tương đối so với nhãn 0; grid nhỏ để hạn chế overfit.\\n    bias = np.zeros(4)\\n    for _ in range(3):\\n        previous = bias.copy()\\n        for cls in [1, 2, 3]:\\n            choices = []\\n            for value in np.linspace(-.4, .4, 17):\\n                proposal = bias.copy()\\n                proposal[cls] = value\\n                objective = macro(y, (scores + proposal).argmax(axis=1)) - .005 * np.square(proposal).sum()\\n                choices.append((objective, -abs(value), float(value)))\\n            bias[cls] = max(choices)[2]\\n        if np.allclose(previous, bias):\\n            break\\n    return bias\\n\\n\\ndef select(scores, y, folds, configs):\\n    # NB-SVM C=1/2 có cảnh báo chưa hội tụ ở thử nghiệm, không chọn chúng.\\n    candidates = [(cfg[\'name\'], {i: 1.}) for i, cfg in enumerate(configs)\\n                  if cfg[\'kind\'] != \'nb\' or cfg[\'C\'] == .5]\\n    # Blends định trước, tránh tìm hàng trăm trọng số trên lớp hiếm.\\n    byname = {c[\'name\']: i for i, c in enumerate(configs)}\\n    for svm_w in [.25, .5, .75]:\\n        candidates.append((f\'blend_svm_nb_{svm_w}\',\\n                           {byname[\'svm_0.5_balanced\']: svm_w, byname[\'nb_0.5\']: 1-svm_w}))\\n    rows = []\\n    for name, weights in candidates:\\n        s = sum(scores[i] * w for i, w in weights.items())\\n        cross_pred = np.zeros(len(y), dtype=int)\\n        for f in np.unique(folds):\\n            fit = folds != f\\n            b = calibrate(s[fit], y[fit])\\n            cross_pred[~fit] = (s[~fit] + b).argmax(axis=1)\\n        raw, calibrated = macro(y, s.argmax(axis=1)), macro(y, cross_pred)\\n        # Chọn bias chỉ nếu calibration cải thiện khi kiểm tra chéo.\\n        use_bias = calibrated > raw + .002\\n        row = {\'name\': name, \'weights\': weights, \'raw_oof_macro_f1\': raw,\\n               \'cross_calibration_macro_f1\': calibrated, \'use_bias\': use_bias,\\n               \'selection_score\': calibrated if use_bias else raw,\\n               \'bias\': calibrate(s, y).tolist() if use_bias else [0., 0., 0., 0.]}\\n        rows.append(row)\\n    rows.sort(key=lambda r: r[\'selection_score\'], reverse=True)\\n    return rows\\n\\n\\nif __name__ == \'__main__\':\\n    folder = Path(\'kaggle_ds102/experiments_v2\')\\n    z = np.load(folder/\'oof.npz\')\\n    cfg = json.loads((folder/\'configs.json\').read_text())\\n    rows = select(z[\'scores\'], z[\'y\'], z[\'folds\'], cfg)\\n    (folder/\'selection.json\').write_text(json.dumps(rows, indent=2))\\n    for r in rows:\\n        print(r[\'name\'], \'raw\', round(r[\'raw_oof_macro_f1\'], 5),\\n              \'cross calibrated\', round(r[\'cross_calibration_macro_f1\'], 5), \'bias\', r[\'bias\'])\\n", "selection_v5.py": "\\"\\"\\"Coarse probability reweighting, validated on separate OOF folds.\\"\\"\\"\\nfrom itertools import product\\nimport numpy as np\\nfrom select_v2 import macro\\n\\n\\ndef fit_weights(probabilities, y):\\n    choices = []\\n    # A small fixed grid: minority-class recall matters for macro F1.\\n    # Identity is included. Never access development or test labels here.\\n    for a,b,c in product([.8,1.,1.2,1.4], [.8,1.,1.2,1.4], [.9,1.,1.1]):\\n        w=np.array([1.,a,b,c])\\n        score=macro(y,(probabilities*w).argmax(1))\\n        penalty=.002*float(np.square(np.log(w)).sum())\\n        choices.append((score-penalty,-float(np.square(np.log(w)).sum()),w))\\n    return max(choices,key=lambda row:row[:2])[2]\\n\\n\\ndef cross_weight(probabilities,y,folds):\\n    pred=np.empty(len(y),dtype=int); weights=[]; fold_deltas=[]\\n    for fold in np.unique(folds):\\n        fit=folds!=fold\\n        w=fit_weights(probabilities[fit],y[fit]); weights.append(w.tolist())\\n        pred[~fit]=(probabilities[~fit]*w).argmax(1)\\n        fold_deltas.append(macro(y[~fit],pred[~fit])-\\n            macro(y[~fit],probabilities[~fit].argmax(1)))\\n    raw=macro(y,probabilities.argmax(1)); cross=macro(y,pred)\\n    use=bool(cross>=raw+.002 and min(fold_deltas)>=-.002)\\n    return {\'raw_macro_f1\':raw,\'cross_weight_macro_f1\':cross,\\n        \'fold_deltas\':fold_deltas,\'fold_weights\':weights,\'use_weights\':use,\\n        \'weights\':fit_weights(probabilities,y).tolist() if use else [1.,1.,1.,1.],\\n        \'selection_score\':cross if use else raw}\\n\\n\\ndef compare(reference,new_oof,y,folds,configs):\\n    rows=[]\\n    row=cross_weight(reference,y,folds)\\n    row.update(model_index=-1,config=\'v3_reference\',new_model_weight=0.)\\n    rows.append(row)\\n    for i,values in enumerate(new_oof):\\n        for alpha in [.15,.3,.5,1.]:\\n            p=(1-alpha)*reference+alpha*values\\n            row=cross_weight(p,y,folds)\\n            row.update(model_index=i,config=configs[i],new_model_weight=alpha)\\n            rows.append(row)\\n    return sorted(rows,key=lambda row:row[\'selection_score\'],reverse=True)\\n", "train_submit_v2.py": "\\"\\"\\"DS102 v2: đặc trưng quan hệ review/sản phẩm, SVM và calibration F1-macro.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport platform\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nimport sklearn\\nfrom scipy import sparse\\nfrom sklearn.metrics import classification_report\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.svm import LinearSVC\\nfrom train_submit import load_data, find_data_dir, make_model, make_features, metrics, LABELS, SEED\\nfrom improved_features import prepare, FeaturesV2, nb_ratios\\nfrom select_v2 import macro, calibrate\\n\\n\\n# Cấu hình được chốt từ cross-validation trên train, trước đánh giá dev v2.\\nRECIPE = [\\n    {\'kind\': \'svm\', \'C\': .5, \'weight\': \'balanced\', \'ensemble_weight\': .75},\\n    {\'kind\': \'nb\', \'C\': .5, \'weight\': \'balanced\', \'ensemble_weight\': .25},\\n]\\n\\n\\ndef fit_predict(fit_df, predict_frames, recipe):\\n    \\"\\"\\"Không nhận nhãn của các tập dự đoán.\\"\\"\\"\\n    y = fit_df.spam_label.to_numpy()\\n    features = FeaturesV2()\\n    Xt, Mt = features.fit_transform(fit_df)\\n    transformed = [features.transform(df) for df in predict_frames]\\n    result = [np.zeros((len(df), 4)) for df in predict_frames]\\n    ratios = nb_ratios(Xt, y) if any(c[\'kind\'] == \'nb\' for c in recipe) else None\\n    for config in recipe:\\n        if config[\'kind\'] == \'svm\':\\n            model = make_model(config, y).fit(sparse.hstack([Xt, Mt], format=\'csr\'), y)\\n            if np.max(model.n_iter_) >= model.max_iter:\\n                raise RuntimeError(\'SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.\')\\n            for j, (Xv, Mv) in enumerate(transformed):\\n                result[j] += config[\'ensemble_weight\'] * model.decision_function(sparse.hstack([Xv, Mv], format=\'csr\'))\\n        elif config[\'kind\'] == \'nb\':\\n            for cls, r in enumerate(ratios):\\n                model = LinearSVC(C=config[\'C\'], class_weight=\'balanced\', dual=True,\\n                                  max_iter=30000, random_state=SEED)\\n                model.fit(sparse.hstack([Xt.multiply(r), Mt], format=\'csr\'), (y == cls).astype(int))\\n                if np.max(model.n_iter_) >= model.max_iter:\\n                    raise RuntimeError(\'NB-SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.\')\\n                for j, (Xv, Mv) in enumerate(transformed):\\n                    result[j][:, cls] += config[\'ensemble_weight\'] * model.decision_function(\\n                        sparse.hstack([Xv.multiply(r), Mv], format=\'csr\'))\\n        else:\\n            raise ValueError(f\\"Model không được hỗ trợ: {config[\'kind\']}\\")\\n    return result\\n\\n\\ndef main_v2(argv=None):\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--data-dir\', default=None)\\n    parser.add_argument(\'--output-dir\', default=\'output_v2\')\\n    parser.add_argument(\'--folds\', type=int, default=3)\\n    args = parser.parse_args(argv)\\n    if args.folds < 2:\\n        parser.error(\'--folds phải >= 2\')\\n    data_dir = find_data_dir(args.data_dir)\\n    data = load_data(data_dir)\\n    expected = {\'comment\', \'product_name\', \'description\', \'category\', \'categories\', \'rating\', \'num_sold\', \'num_review\'}\\n    for name in [\'train\', \'dev\', \'test\']:\\n        missing = expected - set(data[name].columns)\\n        if missing:\\n            raise ValueError(f\'{name} thiếu cột {missing}\')\\n        for col in [\'rating\', \'num_sold\', \'num_review\']:\\n            values = pd.to_numeric(data[name][col], errors=\'raise\')\\n            if not np.isfinite(values).all() or (values < 0).any():\\n                raise ValueError(f\'{name}.{col}: giá trị số không hợp lệ\')\\n            data[name][col] = values\\n    train, dev, test = [prepare(data[n]) for n in [\'train\', \'dev\', \'test\']]\\n    output = Path(args.output_dir).expanduser().resolve()\\n    output.mkdir(parents=True, exist_ok=True)\\n    y = train.spam_label.to_numpy()\\n    oof = np.zeros((len(train), 4))\\n    fold_ids = np.full(len(train), -1)\\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\\n    for fold, (tr, va) in enumerate(splitter.split(train, y, groups=train.comment)):\\n        if set(y[tr]) != set(LABELS):\\n            raise ValueError(\'Fold không đủ 4 lớp; giảm số folds.\')\\n        print(f\'CV fold {fold+1}/{args.folds}\', flush=True)\\n        oof[va] = fit_predict(train.iloc[tr], [train.iloc[va].drop(columns=\'spam_label\')], RECIPE)[0]\\n        fold_ids[va] = fold\\n        print(f\'  Raw F1-macro={macro(y[va], oof[va].argmax(axis=1)):.5f}\', flush=True)\\n    cross_pred = np.zeros(len(train), dtype=int)\\n    for f in range(args.folds):\\n        fit = fold_ids != f\\n        bias_fold = calibrate(oof[fit], y[fit])\\n        cross_pred[~fit] = (oof[~fit] + bias_fold).argmax(axis=1)\\n    raw_score = macro(y, oof.argmax(axis=1))\\n    cross_score = macro(y, cross_pred)\\n    use_bias = cross_score > raw_score + .002\\n    bias = calibrate(oof, y) if use_bias else np.zeros(4)\\n    print(f\'OOF raw={raw_score:.5f}; cross-calibrated={cross_score:.5f}; bias={bias.tolist()}\', flush=True)\\n    print(\'Fit toàn bộ train và dự đoán dev/test...\', flush=True)\\n    dev_scores, test_scores = fit_predict(train, [dev.drop(columns=\'spam_label\'), test], RECIPE)\\n    dev_pred = (dev_scores + bias).argmax(axis=1)\\n    test_pred = (test_scores + bias).argmax(axis=1)\\n    dev_y = dev.spam_label.to_numpy()\\n    unique_dev = ~dev.comment.isin(set(train.comment)).to_numpy()\\n    dev_report = metrics(dev_y, dev_pred)\\n    print(f\\"Dev F1-macro={dev_report[\'macro_f1\']:.5f}\\", flush=True)\\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\\n    by_id = pd.Series(test_pred, index=test.id.astype(str))\\n    submission = data[\'samplesubmission\'].copy()\\n    submission[\'spam_label\'] = submission.ID.astype(str).map(by_id)\\n    if submission.spam_label.isna().any() or not submission.spam_label.isin(LABELS).all():\\n        raise ValueError(\'Nhãn submission không hợp lệ\')\\n    submission[\'spam_label\'] = submission.spam_label.astype(int)\\n    path = output/\'submission_v2.csv\'\\n    submission.to_csv(path, index=False)\\n    saved = pd.read_csv(path)\\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\\n    if len(saved) != len(test) or not saved.ID.equals(data[\'samplesubmission\'].ID):\\n        raise ValueError(\'Sai số dòng hoặc thứ tự ID\')\\n    pd.DataFrame({\'comment\': data[\'dev\'].comment, \'true_label\': dev_y,\\n                  \'predicted_label\': dev_pred, \'comment_seen_in_train\': ~unique_dev}).to_csv(output/\'dev_predictions_v2.csv\', index=False)\\n    np.savez_compressed(output/\'scores_v2.npz\', oof=oof, y=y, folds=fold_ids,\\n                        dev_scores=dev_scores, test_scores=test_scores, bias=bias)\\n    report = {\\n        \'recipe\': RECIPE, \'seed\': SEED, \'folds\': args.folds, \'python\': platform.python_version(),\\n        \'sklearn\': sklearn.__version__, \'data_sha256\': {name: hashlib.sha256((data_dir/file).read_bytes()).hexdigest()\\n            for name, file in [(\'train\',\'ds102_train.csv\'), (\'dev\',\'ds102_dev.csv\'), (\'test\',\'ds102_test.csv\')]},\\n        \'fit_data\': \'train only; no dev or test used to fit features, models, or calibration\',\\n        \'oof_raw_macro_f1\': raw_score, \'oof_cross_calibration_macro_f1\': cross_score,\\n        \'calibration_bias\': bias.tolist(), \'dev\': dev_report,\\n        \'dev_without_comment_overlap\': metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\\n        \'selection_caveat\': \'CV estimates were used for model selection; dev was previously used to inspect baseline errors.\',\\n        \'submission_rows_verified\': len(saved),\\n        \'prediction_counts\': {int(k): int(v) for k, v in saved.spam_label.value_counts().items()},\\n    }\\n    (output/\'metrics_v2.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\\n    print(f\'Đã kiểm tra và lưu {len(saved)} dòng: {path}\', flush=True)\\n    return report\\n\\n\\nif __name__ == \'__main__\':\\n    main_v2()\\n", "catboost_v3.py": "\\"\\"\\"CatBoost text + thông tin review/sản phẩm, không dùng mô hình pretrained.\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\nfrom catboost import CatBoostClassifier\\nfrom improved_features import dense_features, tokens\\nfrom train_submit import SEED\\n\\n\\ndef cat_features(df):\\n    a = dense_features(df)\\n    x = pd.DataFrame(a, columns=[f\'stat_{i}\' for i in range(a.shape[1])])\\n    x[\'comment\'] = [\' \'.join(tokens(t)) or \'emptyreview\' for t in df.comment]\\n    x[\'category\'] = df.category.astype(str).to_numpy()\\n    x[\'rating\'] = df.rating.astype(str).to_numpy()\\n    return x\\n\\n\\ndef make_cat(weight=\'SqrtBalanced\', iterations=500):\\n    return CatBoostClassifier(\\n        iterations=iterations, depth=5, learning_rate=.1, l2_leaf_reg=5,\\n        loss_function=\'MultiClass\', auto_class_weights=weight,\\n        thread_count=4, random_seed=SEED, allow_writing_files=False,\\n        text_features=[\'comment\'], cat_features=[\'category\', \'rating\'],\\n        tokenizers=[{\'tokenizer_id\':\'Space\', \'separator_type\':\'ByDelimiter\', \'delimiter\':\' \'}],\\n        dictionaries=[{\'dictionary_id\':\'Word\', \'gram_order\':\'1\', \'occurrence_lower_bound\':\'3\', \'max_dictionary_size\':\'40000\'},\\n                      {\'dictionary_id\':\'Bigram\', \'gram_order\':\'2\', \'occurrence_lower_bound\':\'3\', \'max_dictionary_size\':\'40000\'}],\\n        feature_calcers=[\'BoW:top_tokens_count=700\', \'NaiveBayes\', \'BM25\'],\\n        verbose=False,\\n    )\\n", "experiment_v6.py": "\\"\\"\\"Hosted Colab: template neighbours and probability-calibrated kernel SVM.\\n\\nNo pretrained models or outside data. All feature fitting is inside the same\\ngrouped train folds as previous versions. Dev labels are not read for selection.\\n\\"\\"\\"\\nimport gc,json,time,traceback,os,platform\\nfrom pathlib import Path\\nimport numpy as np\\nfrom sklearn.preprocessing import normalize\\nfrom sklearn.svm import SVC\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom sklearn.metrics.pairwise import rbf_kernel\\nfrom improved_features import prepare,FeaturesV2\\nfrom train_submit import load_data,metrics,SEED\\nfrom select_v2 import macro\\nfrom selection_v5 import cross_weight\\n\\n\\ndef hosted_guard():\\n    if platform.system()!=\'Linux\' or not os.environ.get(\'COLAB_RELEASE_TAG\'):\\n        raise RuntimeError(\'Hosted Google Colab required; local training disabled.\')\\n\\n\\ndef kernel(text_a,meta_a,text_b,meta_b):\\n    k=(text_a@text_b.T).toarray()\\n    k+=1;np.square(k,out=k)\\n    k+=.5*rbf_kernel(meta_a,meta_b,gamma=.3)\\n    k/=4.5\\n    return np.ascontiguousarray(k,dtype=np.float64)\\n\\n\\ndef neighbours(query,train,y,batch_size=256):\\n    \\"\\"\\"Eight nearest TF-IDF reviews; all references belong to fitting data.\\"\\"\\"\\n    counts=np.bincount(y,minlength=4)\\n    class_weight=np.sqrt(len(y)/(4*counts))\\n    probs=np.zeros((query.shape[0],4),dtype=np.float32)\\n    maximum=np.zeros(query.shape[0],dtype=np.float32)\\n    k=min(8,len(y))\\n    for start in range(0,query.shape[0],batch_size):\\n        sim=(query[start:start+batch_size]@train.T).toarray()\\n        idx=np.argpartition(sim,-k,axis=1)[:,-k:]\\n        values=np.take_along_axis(sim,idx,axis=1)\\n        weights=np.maximum(values-.2,0)**4\\n        labels=y[idx]\\n        votes=np.stack([(weights*(labels==c)).sum(axis=1)*class_weight[c] for c in range(4)],axis=1)\\n        votes+=1e-8\\n        probs[start:start+len(sim)]=votes/votes.sum(axis=1,keepdims=True)\\n        maximum[start:start+len(sim)]=values.max(axis=1)\\n    return probs,maximum\\n\\n\\ndef make_kernel_model(y):\\n    counts=np.bincount(y,minlength=4)\\n    cw={c:float(np.sqrt(len(y)/(4*counts[c]))) for c in range(4)}\\n    # libsvm fits its sigmoid calibrators using only the fit partition.\\n    # Outer grouped CV keeps every held-out review group outside that process.\\n    return SVC(C=1.,kernel=\'precomputed\',class_weight=cw,probability=True,\\n               random_state=SEED,cache_size=512)\\n\\n\\ndef neighbour_class_balance(probabilities,counts,power=.5):\\n    \\"\\"\\"Change neighbour voting\'s class-count correction without refitting IDF.\\"\\"\\"\\n    factor=(np.asarray(counts,dtype=float)/max(counts))**(.5-power)\\n    p=probabilities*factor\\n    return p/p.sum(axis=1,keepdims=True)\\n\\n\\ndef candidates(base,hard,prob,near,similarity):\\n    yield \'v3_reference\',base,{\'kind\':\'reference\',\'alpha\':0.}\\n    for alpha in [.05,.1,.2]:\\n        yield f\'hard_kernel_{alpha}\',(1-alpha)*base+alpha*np.eye(4)[hard],{\'kind\':\'hard\',\'alpha\':alpha}\\n    for alpha in [.15,.3,.5]:\\n        yield f\'probability_kernel_{alpha}\',(1-alpha)*base+alpha*prob,{\'kind\':\'probability\',\'alpha\':alpha}\\n    for threshold in [.7,.85]:\\n        for alpha in [.3,.5]:\\n            gate=(similarity>=threshold).astype(float)[:,None]*alpha\\n            yield f\'neighbours_{threshold}_{alpha}\',(1-gate)*base+gate*near,{\'kind\':\'neighbours\',\'alpha\':alpha,\'threshold\':threshold}\\n\\n\\ndef select(base,hard,prob,near,similarity,y,folds):\\n    rows=[]\\n    configurations=list(candidates(base,hard,prob,near,similarity))\\n    # The original inverse-square-root class correction is compared with\\n    # milder/no correction, using only fit-fold label counts.\\n    for power in [0.,.25]:\\n        adjusted=np.empty_like(near)\\n        for fold in np.unique(folds):\\n            mask=folds==fold;counts=np.bincount(y[~mask],minlength=4)\\n            adjusted[mask]=neighbour_class_balance(near[mask],counts,power)\\n        gate=.3*(similarity>=.85).astype(float)[:,None]\\n        configurations.append((f\'neighbours_0.85_0.3_power_{power}\',(1-gate)*base+gate*adjusted,\\n            {\'kind\':\'neighbours\',\'alpha\':.3,\'threshold\':.85,\'class_power\':power}))\\n    for name,p,recipe in configurations:\\n        row=cross_weight(p,y,folds)\\n        raw_folds=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\\n        row[\'fold_macro_f1\']=[v+d for v,d in zip(raw_folds,row[\'fold_deltas\'])] if row[\'use_weights\'] else raw_folds\\n        row.update(name=name,recipe=recipe)\\n        rows.append(row)\\n    rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\\n    baseline=next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\')\\n    for row in rows:\\n        row[\'fold_delta_vs_v5\']=(np.array(row[\'fold_macro_f1\'])-np.array(baseline[\'fold_macro_f1\'])).tolist()\\n    return rows\\n\\n\\ndef choose_stable(rows):\\n    \\"\\"\\"Prefer consistent fold gains when aggregate CV differs by < .001.\\"\\"\\"\\n    baseline=next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\')\\n    best=rows[0]\\n    if best[\'selection_score\']<baseline[\'selection_score\']+.002:return baseline\\n    stable=[r for r in rows if r[\'recipe\'][\'kind\']!=\'reference\'\\n            and r[\'selection_score\']>=best[\'selection_score\']-.001\\n            and min(r[\'fold_delta_vs_v5\'])>=0]\\n    return max(stable,key=lambda r:r[\'selection_score\']) if stable else best\\n\\n\\ndef main(data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6_experiment\',\\n         reference_path=\'/content/colab_reference.npz\'):\\n    hosted_guard()\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(stage,**values):\\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\\n    try:\\n        df=prepare(load_data(Path(data_dir))[\'train\']);y=df.spam_label.to_numpy()\\n        ref=np.load(reference_path);assert np.array_equal(y,ref[\'y\'])\\n        hard=np.full(len(y),-1,dtype=int);prob=np.zeros((len(y),4),dtype=np.float32)\\n        near=np.zeros_like(prob);similarity=np.zeros(len(y),dtype=np.float32)\\n        folds=np.full(len(y),-1);results=[]\\n        split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\\n        for fold,(tr,va) in enumerate(split.split(df,y,groups=df.comment)):\\n            folds[va]=fold;status(\'features\',fold=fold+1,results=results)\\n            f=FeaturesV2();xt,mt=f.fit_transform(df.iloc[tr])\\n            xv,mv=f.transform(df.iloc[va].drop(columns=\'spam_label\'))\\n            xt=normalize(xt);xv=normalize(xv)\\n            mt.data=np.clip(mt.data,-.75,.75);mv.data=np.clip(mv.data,-.75,.75)\\n            status(\'neighbours\',fold=fold+1,results=results)\\n            near[va],similarity[va]=neighbours(xv,xt,y[tr])\\n            status(\'kernel_matrix\',fold=fold+1,results=results)\\n            kt=kernel(xt,mt,xt,mt);kv=kernel(xv,mv,xt,mt)\\n            status(\'kernel_probability_fit\',fold=fold+1,results=results)\\n            m=make_kernel_model(y[tr]);m.fit(kt,y[tr])\\n            assert np.array_equal(m.classes_,np.arange(4))\\n            hard[va]=m.predict(kv);prob[va]=m.predict_proba(kv)\\n            results.append({\'kernel_hard\':macro(y[va],hard[va]),\\n                            \'kernel_prob\':macro(y[va],prob[va].argmax(1)),\\n                            \'neighbours\':macro(y[va],near[va].argmax(1))})\\n            del m,kt,kv,xt,xv,mt,mv,f;gc.collect()\\n            np.savez_compressed(out/\'partial.npz\',hard=hard,prob=prob,near=near,\\n                                similarity=similarity,y=y,folds=folds)\\n        assert np.array_equal(folds,ref[\'folds\'])\\n        np.savez_compressed(out/\'oof.npz\',hard=hard,prob=prob,near=near,\\n                            similarity=similarity,y=y,folds=folds)\\n        rows=select(ref[\'oof\'],hard,prob,near,similarity,y,folds)\\n        (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\\n        report={\'fold_scores\':results,\'best\':rows[0],\'chosen\':choose_stable(rows),\\n            \'selection_rule\':\'Require >=.002 CV gain; within .001 of best, prefer a candidate improving every fold over v5.\',\\n            \'reference_v5\':next(r for r in rows if r[\'recipe\'][\'kind\']==\'reference\'),\\n            \'pretrained\':False,\'fit_data\':\'train only; dev not read for selection\',\\n            \'execution\':\'hosted Colab CPU\',\\n            \'caveat\':\'Repeated OOF model selection; weight calibration on existing OOF is not fully nested CV.\'}\\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2));status(\'complete\',**report)\\n        return report\\n    except BaseException as exc:\\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\\n", "latent_v6.py": "\\"\\"\\"Train-only latent TF-IDF review/product relations + numeric CatBoost.\\n\\nTruncatedSVD is fitted from scratch inside each training fold. No pretrained\\nembeddings, language models or outside corpora are used.\\n\\"\\"\\"\\nimport gc,json,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy import sparse\\nfrom sklearn.decomposition import TruncatedSVD\\nfrom sklearn.preprocessing import normalize\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom catboost import CatBoostClassifier\\nfrom improved_features import FeaturesV2,dense_features,prepare\\nfrom experiment_v6 import hosted_guard\\nfrom selection_v5 import cross_weight\\nfrom select_v2 import macro\\nfrom train_submit import load_data,SEED\\n\\n\\nclass LatentFeatures:\\n    def __init__(self):\\n        self.base=FeaturesV2()\\n        self.svd=TruncatedSVD(n_components=128,n_iter=5,random_state=SEED)\\n\\n    def transform(self,df,fit=False):\\n        if fit:\\n            review=sparse.hstack([self.base.word.fit_transform(df.comment),\\n                                  self.base.char.fit_transform(df.comment)],format=\'csr\')\\n            review=normalize(review)\\n            z=self.svd.fit_transform(review)\\n        else:\\n            review=sparse.hstack([self.base.word.transform(df.comment),\\n                                  self.base.char.transform(df.comment)],format=\'csr\')\\n            review=normalize(review)\\n            z=self.svd.transform(review)\\n        title=normalize(sparse.hstack([self.base.word.transform(df.product_name),\\n                                      self.base.char.transform(df.product_name)],format=\'csr\'))\\n        t=self.svd.transform(title)\\n        cosine=np.asarray(review.multiply(title).sum(axis=1)).ravel()\\n        x=np.hstack([z,t,z*t,np.abs(z-t),cosine[:,None],dense_features(df)])\\n        x=pd.DataFrame(x.astype(np.float32),columns=[f\'f_{i}\' for i in range(x.shape[1])])\\n        x[\'category\']=df.category.astype(str).to_numpy()\\n        x[\'rating\']=df.rating.astype(str).to_numpy()\\n        return x\\n\\n\\ndef make_model():\\n    return CatBoostClassifier(iterations=800,depth=5,learning_rate=.06,l2_leaf_reg=8,\\n        loss_function=\'MultiClass\',auto_class_weights=\'SqrtBalanced\',\\n        cat_features=[\'category\',\'rating\'],max_ctr_complexity=1,\\n        random_seed=SEED,thread_count=2,allow_writing_files=False,verbose=False)\\n\\n\\ndef main(data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6_latent\',reference_path=\'/content/colab_reference.npz\'):\\n    hosted_guard()\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(stage,**values):\\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\\n    try:\\n        df=prepare(load_data(Path(data_dir))[\'train\']);y=df.spam_label.to_numpy()\\n        ref=np.load(reference_path);assert np.array_equal(y,ref[\'y\'])\\n        trees=[400,800];oof=np.zeros((2,len(y),4),dtype=np.float32)\\n        folds=np.full(len(y),-1);results=[[],[]]\\n        split=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED)\\n        for f,(tr,va) in enumerate(split.split(df,y,groups=df.comment)):\\n            folds[va]=f;status(\'fit_svd\',fold=f+1,results=results)\\n            features=LatentFeatures();xt=features.transform(df.iloc[tr],fit=True)\\n            xv=features.transform(df.iloc[va].drop(columns=\'spam_label\'))\\n            status(\'fit_catboost\',fold=f+1,results=results)\\n            m=make_model();m.fit(xt,y[tr],verbose=False)\\n            for i,n in enumerate(trees):\\n                oof[i,va]=m.predict_proba(xv,ntree_end=n)\\n                results[i].append(macro(y[va],oof[i,va].argmax(1)))\\n            del features,m,xt,xv;gc.collect()\\n            np.savez_compressed(out/\'partial.npz\',oof=oof,y=y,folds=folds)\\n        assert np.array_equal(folds,ref[\'folds\'])\\n        rows=[]\\n        for i,n in enumerate(trees):\\n            for alpha in [.15,.3,.5]:\\n                p=(1-alpha)*ref[\'oof\']+alpha*oof[i]\\n                row=cross_weight(p,y,folds)\\n                raw=[macro(y[folds==f],p[folds==f].argmax(1)) for f in np.unique(folds)]\\n                row[\'fold_macro_f1\']=[a+b for a,b in zip(raw,row[\'fold_deltas\'])] if row[\'use_weights\'] else raw\\n                row.update(name=f\'latent_cat_{n}_{alpha}\',recipe={\'kind\':\'latent\',\'trees\':n,\'alpha\':alpha})\\n                rows.append(row)\\n        rows.sort(key=lambda r:r[\'selection_score\'],reverse=True)\\n        np.savez_compressed(out/\'oof.npz\',oof=oof,y=y,folds=folds)\\n        (out/\'selection.json\').write_text(json.dumps(rows,indent=2))\\n        report={\'best\':rows[0],\'fold_scores\':results,\'pretrained\':False,\'fit_data\':\'train only\',\\n                \'representation\':\'TF-IDF and 64D SVD fitted on each training fold; review/title relations\',\\n                \'caveat\':\'Repeated OOF selection; weighting checks not fully nested CV.\'}\\n        (out/\'cv_results.json\').write_text(json.dumps(report,indent=2));status(\'complete\',**report)\\n        return report\\n    except BaseException as exc:\\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\\n", "train_submit_v6.py": "\\"\\"\\"Train-only DS102 v6 inference; hosted Colab required; no pretrained.\\"\\"\\"\\nimport gc,hashlib,json,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.special import softmax\\nfrom sklearn.preprocessing import normalize\\nfrom improved_features import prepare,FeaturesV2\\nfrom experiment_v6 import hosted_guard,kernel,neighbours,make_kernel_model,neighbour_class_balance\\nfrom train_submit import load_data,metrics\\n\\n\\ndef baseline_predict(train,frames):\\n    from train_submit_v2 import fit_predict,RECIPE\\n    from catboost_v3 import make_cat,cat_features\\n    margins=fit_predict(train,frames,RECIPE);gc.collect()\\n    m=make_cat(\'SqrtBalanced\',iterations=500);m.set_params(thread_count=2)\\n    m.fit(cat_features(train),train.spam_label.to_numpy(),verbose=250)\\n    return [.5*softmax(3*s,axis=1)+.5*m.predict_proba(cat_features(d)) for s,d in zip(margins,frames)]\\n\\n\\ndef component_predict(train,frames,recipe,status=None):\\n    hosted_guard()\\n    if recipe[\'kind\']==\'latent\':\\n        from latent_v6 import LatentFeatures,make_model as make_latent_model\\n        if status:status(\'fit_latent_features\')\\n        f=LatentFeatures();xt=f.transform(train,fit=True)\\n        if status:status(\'fit_latent_catboost\')\\n        m=make_latent_model();m.fit(xt,train.spam_label.to_numpy(),verbose=False)\\n        return [(m.predict_proba(f.transform(df),ntree_end=recipe[\'trees\']),None) for df in frames]\\n    f=FeaturesV2();xt,mt=f.fit_transform(train)\\n    xt=normalize(xt);mt.data=np.clip(mt.data,-.75,.75)\\n    y=train.spam_label.to_numpy();kind=recipe[\'kind\']\\n    if kind==\'neighbours\':\\n        result=[]\\n        for df in frames:\\n            x,_=f.transform(df);p,s=neighbours(normalize(x),xt,y)\\n            if recipe.get(\'class_power\',.5)!=.5:\\n                p=neighbour_class_balance(p,np.bincount(y,minlength=4),recipe[\'class_power\'])\\n            result.append((p,s))\\n        return result\\n    if status:status(\'full_kernel\')\\n    gram=kernel(xt,mt,xt,mt)\\n    m=make_kernel_model(y)\\n    if kind==\'hard\':m.set_params(probability=False)\\n    if status:status(\'full_kernel_fit\')\\n    m.fit(gram,y);del gram;gc.collect()\\n    result=[]\\n    for frame_idx,df in enumerate(frames):\\n        if status:status(\'predict\',frame=frame_idx)\\n        pieces=[]\\n        for start in range(0,len(df),256):\\n            xv,mv=f.transform(df.iloc[start:start+256]);xv=normalize(xv)\\n            mv.data=np.clip(mv.data,-.75,.75)\\n            gram=kernel(xv,mv,xt,mt)\\n            pieces.append(np.eye(4)[m.predict(gram)] if kind==\'hard\' else m.predict_proba(gram))\\n        result.append((np.vstack(pieces),None))\\n    return result\\n\\n\\ndef main(selection,data_dir=\'/content/ds102_data\',output_dir=\'/content/ds102_v6\',reference_path=None):\\n    hosted_guard()\\n    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)\\n    def status(stage,**values):\\n        (out/\'status.json\').write_text(json.dumps(dict(stage=stage,time=time.time(),**values),indent=2))\\n    try:\\n        data=load_data(Path(data_dir))\\n        train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        frames=[dev.drop(columns=\'spam_label\'),test]\\n        recipe=selection[\'recipe\']\\n        if recipe[\'kind\']==\'reference\':raise ValueError(\'v6 experiment did not choose a new component; retain v5.\')\\n        status(\'component_features\',recipe=recipe)\\n        extra=component_predict(train,frames,recipe,status)\\n        gc.collect();status(\'baseline_predictions\')\\n        if reference_path:\\n            z=np.load(reference_path)\\n            assert np.array_equal(z[\'y\'],train.spam_label.to_numpy())\\n            base=[z[\'dev\'],z[\'test\']]\\n            baseline_source=\'previously user-authorized cached v3 predictions\'\\n        else:\\n            base=baseline_predict(train,frames)\\n            baseline_source=\'fresh train-only fit on hosted Colab\'\\n        scores=[];alpha=float(recipe[\'alpha\'])\\n        for b,(p,s) in zip(base,extra):\\n            weight=alpha\\n            if recipe[\'kind\']==\'neighbours\':weight=alpha*(s>=recipe[\'threshold\']).astype(float)[:,None]\\n            scores.append((1-weight)*b+weight*p)\\n        weights=np.asarray(selection[\'weights\'])\\n        dp,tp=[(p*weights).argmax(1) for p in scores]\\n        sub=data[\'samplesubmission\'].copy()\\n        lookup=pd.Series(tp,index=test.id.astype(str))\\n        sub[\'spam_label\']=sub.ID.astype(str).map(lookup)\\n        if sub.spam_label.isna().any() or not sub.spam_label.isin([0,1,2,3]).all():raise ValueError(\'Invalid IDs/labels\')\\n        sub.spam_label=sub.spam_label.astype(int)\\n        path=out/\'submission_v11_experimental.csv\';sub.to_csv(path,index=False)\\n        saved=pd.read_csv(path);pd.testing.assert_frame_equal(saved,sub,check_dtype=False)\\n        assert len(saved)==len(test) and saved.ID.equals(data[\'samplesubmission\'].ID)\\n        np.savez_compressed(out/\'predictions.npz\',dev=scores[0],test=scores[1],weights=weights)\\n        report={\'selection\':selection,\'dev\':metrics(dev.spam_label.to_numpy(),dp),\\n            \'v5_reference\':{\'cross_weight_oof_macro_f1\':.7681613908279921,\'dev_macro_f1\':.7348200162618499,\'user_reported_kaggle\':.76740},\\n            \'rows_verified\':len(saved),\'pretrained\':False,\'fit_data\':\'train only; dev/test excluded from feature/model fitting\',\\n            \'selection_data\':\'grouped train OOF for candidate hyperparameters; dev used to compare finalists\',\\n            \'new_training_execution\':\'hosted Google Colab CPU\',\'baseline_source\':baseline_source,\\n            \'libraries\':{\'scikit-learn\':\'1.7.2; BSD-3-Clause\',\'CatBoost\':\'1.2.10; Apache-2.0\'},\\n            \'data_sha256\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(data_dir).glob(\'ds102_*.csv\')},\\n            \'caveat\':\'Repeated OOF selection; weighting cross-validation is on existing OOF, not fully nested. Dev reused in development; Kaggle score unknown.\'}\\n        (out/\'metrics_v11_experimental.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        status(\'complete\',dev_macro_f1=report[\'dev\'][\'macro_f1\'],rows=len(saved))\\n        return report\\n    except BaseException as exc:\\n        status(\'error\',error=str(exc),traceback=traceback.format_exc());raise\\n", "cloud_checks.py": "import os,platform\\nimport numpy as np\\ndef guard():\\n    if platform.system()!=\'Linux\' or not os.environ.get(\'COLAB_RELEASE_TAG\'):\\n        raise RuntimeError(\'Use hosted Google Colab, not a local runtime.\')\\ndef checked(p,n):\\n    p=np.asarray(p)\\n    assert p.shape==(n,4) and np.isfinite(p).all()\\n    assert (p>=0).all() and np.allclose(p.sum(1),1,atol=1e-5)\\n    return p\\n", "catalogue_svd_v40.py": "\\"\\"\\"Train-fold catalogue vocabulary/SVD, independent of review-only SVD128.\\n\\nOnly supplied fitting product metadata form the corpus. No pretrained parameters,\\nexternal data, development/test fitting, or labels enter this representation.\\n\\"\\"\\"\\nimport gc,html,json,re\\nfrom pathlib import Path\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy import sparse\\nfrom sklearn.decomposition import TruncatedSVD\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.preprocessing import normalize\\nfrom latent_v6 import LatentFeatures,make_model\\nfrom cloud_checks import guard,checked\\n\\nCONFIG=dict(dimensions=32,description_char_cap=4096,word_max_features=50000,\\n    char_max_features=80000,word_ngram=[1,2],char_ngram=[3,5],max_df=.95,\\n    trees=800,depth=5,lr=.06,l2=8,seed=102)\\n\\ndef catalogue_documents(df):\\n    def clean(value):\\n        value=html.unescape(str(value))\\n        value=re.sub(r\'<[^>]+>\',\' \',value)\\n        return re.sub(r\'\\\\s+\',\' \',value).strip()[:CONFIG[\'description_char_cap\']]\\n    return [str(t)+\' \'+clean(d) for t,d in zip(df.product_name,df.description)]\\n\\nclass CatalogueEncoder:\\n    def __init__(self):\\n        self.word=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_df=.95,\\n            max_features=50000,sublinear_tf=True,dtype=np.float32)\\n        self.char=TfidfVectorizer(analyzer=\'char\',ngram_range=(3,5),min_df=3,\\n            max_df=.95,max_features=80000,sublinear_tf=True,dtype=np.float32)\\n    def vectorize(self,texts):\\n        return normalize(sparse.hstack([self.word.transform(texts),\\n            self.char.transform(texts)],format=\'csr\'))\\n    def fit(self,train):\\n        guard();docs=list(dict.fromkeys(catalogue_documents(train)))\\n        assert len(docs)>3,\'Need at least four distinct fitting catalogue documents\'\\n        x=normalize(sparse.hstack([self.word.fit_transform(docs),\\n            self.char.fit_transform(docs)],format=\'csr\'))\\n        self.k=min(CONFIG[\'dimensions\'],x.shape[0]-1,x.shape[1]-1)\\n        assert self.k>0\\n        self.svd=TruncatedSVD(n_components=self.k,n_iter=5,random_state=102)\\n        self.svd.fit(x);self.fit_documents=len(docs);self.vocabulary_size=x.shape[1]\\n        return self\\n    def transform(self,df):\\n        r=self.vectorize(df.comment.tolist());p=self.vectorize(catalogue_documents(df))\\n        z=self.svd.transform(r);t=self.svd.transform(p)\\n        zn=np.linalg.norm(z,axis=1);tn=np.linalg.norm(t,axis=1)\\n        lexical=np.asarray(r.multiply(p).sum(axis=1)).ravel()\\n        cosine=(z*t).sum(1)/np.maximum(zn*tn,1e-9)\\n        z=np.pad(z,((0,0),(0,32-self.k)));t=np.pad(t,((0,0),(0,32-self.k)))\\n        out=np.hstack([z,t,z*t,np.abs(z-t),lexical[:,None],\\n            cosine[:,None],zn[:,None],tn[:,None]]).astype(np.float32)\\n        assert out.shape==(len(df),132) and np.isfinite(out).all()\\n        return out\\n\\ndef fit_predict(train,frames,folder,feature_id,status,cache_path=None,cache_id=None):\\n    guard();folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\\n    if cache_path is not None:\\n        q=np.load(cache_path,allow_pickle=False);assert q[\'feature_id\'].item()==cache_id\\n        cols=q[\'numeric_columns\'][:575].tolist();assert cols==[f\'f_{i}\' for i in range(575)]\\n        all_frames=[]\\n        for prefix in [\'train\']+[f\'frame{i}\' for i in range(len(frames))]:\\n            x=pd.DataFrame(q[prefix+\'_numeric\'][:,:575],columns=cols)\\n            for j,c in enumerate([\'category\',\'rating\']):x[c]=q[prefix+\'_categories\'][:,j]\\n            all_frames.append(x)\\n    else:\\n        status(\'fit_full_svd128_features\')\\n        f=LatentFeatures();f.svd.set_params(n_components=128,random_state=102)\\n        all_frames=[f.transform(train,fit=True)]+[f.transform(d) for d in frames]\\n        del f;gc.collect()\\n    status(\'fit_catalogue_only_svd32\')\\n    encoder=CatalogueEncoder().fit(train)\\n    joblib.dump(encoder,folder/\'catalogue_encoder.joblib\',compress=3)\\n    augmented=[]\\n    for x,df in zip(all_frames,[train]+frames):\\n        assert len(x)==len(df)\\n        assert np.array_equal(x[[\'category\',\'rating\']].to_numpy(dtype=str),\\n            df[[\'category\',\'rating\']].astype(str).to_numpy())\\n        extra=encoder.transform(df)\\n        joined=pd.concat([x.reset_index(drop=True),pd.DataFrame(extra,\\n            columns=[f\'catalogue_{i}\' for i in range(extra.shape[1])])],axis=1)\\n        assert joined.shape[1]==709\\n        augmented.append(joined)\\n    status(\'fit_catboost_catalogue_svd\',feature_columns=709,\\n        fitting_catalogue_documents=encoder.fit_documents,\\n        catalogue_vocabulary=encoder.vocabulary_size)\\n    m=make_model();m.fit(augmented[0],train.spam_label.to_numpy(),verbose=200)\\n    assert np.array_equal(m.classes_,[0,1,2,3])\\n    result=[checked(m.predict_proba(x),len(x)) for x in augmented[1:]]\\n    m.save_model(str(folder/\'catboost.cbm\'))\\n    (folder/\'metadata.json\').write_text(json.dumps(dict(config=CONFIG,feature_id=feature_id,\\n        fit_documents=encoder.fit_documents,vocabulary_size=encoder.vocabulary_size,\\n        fitted_dimensions=encoder.k),indent=2))\\n    del m,encoder,all_frames,augmented;gc.collect();return result\\n", "lexical_chi2_v41.py": "\\"\\"\\"Train-only classwise lexical selection plus original SVD128 CatBoost.\\n\\nUnlike frequency-limited text dictionaries, allocate a fixed selection quota\\nto each class. Feature selection uses fitting labels only; transform ignores\\nlabels. Every parameter is learned from the supplied training partition.\\n\\"\\"\\"\\nimport gc,json\\nfrom pathlib import Path\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.feature_extraction.text import CountVectorizer\\nfrom sklearn.feature_selection import chi2\\nfrom latent_v6 import LatentFeatures,make_model\\nfrom cloud_checks import guard,checked\\n\\nCONFIG=dict(per_class_per_view=128,word_max_features=100000,char_max_features=120000,\\n    min_df=3,word_ngram=[1,3],char_ngram=[2,5],binary=True,\\n    trees=800,depth=5,lr=.06,l2=8,seed=102,new_latent_coefficient=.15)\\n\\nclass LexicalEncoder:\\n    def __init__(self):\\n        self.word=CountVectorizer(ngram_range=(1,3),min_df=3,\\n            max_features=100000,binary=True,dtype=np.float32)\\n        self.char=CountVectorizer(analyzer=\'char\',ngram_range=(2,5),min_df=3,\\n            max_features=120000,binary=True,dtype=np.float32)\\n    def fit(self,train):\\n        guard();y=train.spam_label.to_numpy();assert np.array_equal(np.unique(y),[0,1,2,3])\\n        self.indices=[];self.vocabulary_sizes=[]\\n        for vectorizer in [self.word,self.char]:\\n            x=vectorizer.fit_transform(train.comment)\\n            selected=set()\\n            for label in range(4):\\n                scores=chi2(x,(y==label).astype(int))[0]\\n                scores=np.nan_to_num(scores,nan=-np.inf)\\n                selected.update(np.argsort(-scores,kind=\'stable\')[:128].tolist())\\n            self.indices.append(np.array(sorted(selected),dtype=np.int64))\\n            self.vocabulary_sizes.append(x.shape[1])\\n        self.selected_columns=sum(len(i) for i in self.indices)\\n        return self\\n    def transform(self,df):\\n        out=np.hstack([v.transform(df.comment)[:,indices].toarray()\\n            for v,indices in zip([self.word,self.char],self.indices)]).astype(np.float32)\\n        assert out.shape==(len(df),self.selected_columns)\\n        assert np.isfinite(out).all() and np.isin(out,[0.,1.]).all()\\n        return out\\n\\ndef fit_predict(train,frames,folder,feature_id,status,cache_path=None,cache_id=None):\\n    guard();folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\\n    if cache_path is not None:\\n        q=np.load(cache_path,allow_pickle=False);assert q[\'feature_id\'].item()==cache_id\\n        cols=q[\'numeric_columns\'][:575].tolist();assert cols==[f\'f_{i}\' for i in range(575)]\\n        all_frames=[]\\n        for prefix in [\'train\']+[f\'frame{i}\' for i in range(len(frames))]:\\n            x=pd.DataFrame(q[prefix+\'_numeric\'][:,:575],columns=cols)\\n            for j,c in enumerate([\'category\',\'rating\']):x[c]=q[prefix+\'_categories\'][:,j]\\n            all_frames.append(x)\\n    else:\\n        status(\'fit_full_svd128_features\')\\n        f=LatentFeatures();f.svd.set_params(n_components=128,random_state=102)\\n        all_frames=[f.transform(train,fit=True)]+[f.transform(d) for d in frames]\\n        del f;gc.collect()\\n    status(\'fit_train_only_classwise_lexical_selection\')\\n    encoder=LexicalEncoder().fit(train)\\n    joblib.dump(encoder,folder/\'lexical_selector.joblib\',compress=3)\\n    augmented=[]\\n    for x,df in zip(all_frames,[train]+frames):\\n        assert len(x)==len(df)\\n        assert np.array_equal(x[[\'category\',\'rating\']].to_numpy(dtype=str),\\n            df[[\'category\',\'rating\']].astype(str).to_numpy())\\n        extra=encoder.transform(df)\\n        joined=pd.concat([x.reset_index(drop=True),pd.DataFrame(extra,\\n            columns=[f\'chi2_{i}\' for i in range(extra.shape[1])])],axis=1)\\n        assert joined.shape[1]==577+encoder.selected_columns\\n        augmented.append(joined)\\n    status(\'fit_catboost_selected_lexical\',feature_columns=augmented[0].shape[1],\\n        selected_lexical_columns=encoder.selected_columns,\\n        original_vocabulary_sizes=encoder.vocabulary_sizes)\\n    m=make_model();m.fit(augmented[0],train.spam_label.to_numpy(),verbose=200)\\n    assert np.array_equal(m.classes_,[0,1,2,3])\\n    result=[checked(m.predict_proba(x),len(x)) for x in augmented[1:]]\\n    m.save_model(str(folder/\'catboost.cbm\'))\\n    (folder/\'metadata.json\').write_text(json.dumps(dict(config=CONFIG,feature_id=feature_id,\\n        selected_columns=encoder.selected_columns,vocabulary_sizes=encoder.vocabulary_sizes),indent=2))\\n    del m,encoder,all_frames,augmented;gc.collect();return result\\n", "review_common.py": "\\"\\"\\"Shared helpers for the Claude review experiments (no fitting in this file).\\n\\nNear-duplicate grouping uses exact token-set Jaccard on a binary token matrix\\nwhose vocabulary is just \\"tokens seen in the given texts\\" (deterministic\\ncounting, not a learned model). Still: call these from Colab scripts.\\n\\"\\"\\"\\nimport re\\nimport numpy as np\\nfrom scipy import sparse\\n\\nTOK = re.compile(r\'(?u)\\\\b\\\\w+\\\\b\')\\nW_V5 = np.array([1., 1., 1.2, .9])\\nW_V6 = np.array([1., 1.2, 1.2, .9])\\n\\n\\ndef token_matrix(texts, vocab=None):\\n    \\"\\"\\"Binary doc x token matrix. If vocab is given, unseen tokens still count in set sizes.\\"\\"\\"\\n    grow = vocab is None\\n    vocab = {} if grow else vocab\\n    rows, cols, sizes = [], [], []\\n    for i, t in enumerate(texts):\\n        s = set(TOK.findall(t)); sizes.append(len(s))\\n        for w in s:\\n            j = vocab.setdefault(w, len(vocab)) if grow else vocab.get(w)\\n            if j is not None:\\n                rows.append(i); cols.append(j)\\n    b = sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(len(texts), len(vocab)))\\n    return b, np.asarray(sizes, np.float32), vocab\\n\\n\\ndef _find(parent, i):\\n    while parent[i] != i:\\n        parent[i] = parent[parent[i]]; i = parent[i]\\n    return i\\n\\n\\ndef near_dup_groups(texts, thr=.6, min_tokens=4, chunk=1024):\\n    \\"\\"\\"Connected components of (exact duplicate) OR (Jaccard >= thr and both have >= min_tokens tokens).\\"\\"\\"\\n    texts = list(texts); n = len(texts)\\n    parent = np.arange(n)\\n    def union(a, b):\\n        ra, rb = _find(parent, a), _find(parent, b)\\n        if ra != rb: parent[max(ra, rb)] = min(ra, rb)\\n    first = {}\\n    for i, t in enumerate(texts):\\n        if t in first: union(first[t], i)\\n        else: first[t] = i\\n    b, s, _ = token_matrix(texts); bt = b.T.tocsc(); ok = s >= min_tokens\\n    for start in range(0, n, chunk):\\n        end = min(start + chunk, n)\\n        inter = (b[start:end] @ bt).toarray()\\n        jac = inter / np.maximum(s[start:end, None] + s[None, :] - inter, 1)\\n        ii, jj = np.nonzero(jac >= thr); ii = ii + start\\n        keep = (jj > ii) & ok[ii] & ok[jj]\\n        for a, c in zip(ii[keep], jj[keep]): union(int(a), int(c))\\n    roots = np.array([_find(parent, i) for i in range(n)])\\n    _, gid = np.unique(roots, return_inverse=True)\\n    return gid\\n\\n\\ndef max_jaccard(query_texts, ref_texts, chunk=1024):\\n    \\"\\"\\"For each query text: max token-set Jaccard to any reference text (0 if no refs).\\"\\"\\"\\n    if len(ref_texts) == 0: return np.zeros(len(query_texts), np.float32)\\n    rb, rs, vocab = token_matrix(ref_texts); qb, qs, _ = token_matrix(query_texts, vocab)\\n    rbt = rb.T.tocsc(); out = np.zeros(len(query_texts), np.float32)\\n    for start in range(0, len(query_texts), chunk):\\n        end = min(start + chunk, len(query_texts))\\n        inter = (qb[start:end] @ rbt).toarray()\\n        out[start:end] = (inter / np.maximum(qs[start:end, None] + rs[None, :] - inter, 1)).max(1)\\n    return out\\n\\n\\ndef class_f1(y, pred, w=None):\\n    cm = np.bincount(4 * y + pred, weights=w, minlength=16).reshape(4, 4)\\n    den = cm.sum(0) + cm.sum(1)\\n    return np.divide(2 * np.diag(cm), den, out=np.zeros(4), where=den > 0)\\n\\n\\ndef macro(y, pred, w=None):\\n    return float(class_f1(y, pred, w).mean())\\n\\n\\ndef group_bootstrap_delta(y, pred_a, pred_b, groups, n=1000, seed=102):\\n    \\"\\"\\"Cluster bootstrap of macro(pred_b) - macro(pred_a). Returns mean, 5%, 95%, P(delta>0).\\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    _, inv = np.unique(groups, return_inverse=True); g = inv.max() + 1\\n    d = np.empty(n)\\n    for k in range(n):\\n        w = np.bincount(rng.integers(0, g, g), minlength=g)[inv].astype(float)\\n        d[k] = macro(y, pred_b, w) - macro(y, pred_a, w)\\n    return {\'mean\': float(d.mean()), \'q05\': float(np.quantile(d, .05)),\\n            \'q95\': float(np.quantile(d, .95)), \'p_gt_0\': float(np.mean(d > 0))}\\n\\n\\ndef class1_strata(y, pred, template_mask):\\n    \\"\\"\\"Class-1 F1 on rows that look like a class-1 template in the fitting data vs. novel rows.\\"\\"\\"\\n    out = {}\\n    for name, m in [(\'template\', template_mask), (\'novel\', ~template_mask)]:\\n        tp = int(np.sum((y == 1) & (pred == 1) & m)); fn = int(np.sum((y == 1) & (pred != 1) & m))\\n        fp = int(np.sum((y != 1) & (pred == 1) & m))\\n        out[name] = {\'pos\': tp + fn, \'tp\': tp, \'fp\': fp, \'fn\': fn,\\n                     \'f1\': 2 * tp / max(2 * tp + fp + fn, 1), \'recall\': tp / max(tp + fn, 1)}\\n    return out\\n", "codex_checked_common.py": "\\"\\"\\"Checked statistical helpers; no model or vectorizer fitting.\\"\\"\\"\\nimport numpy as np\\nfrom scipy.special import softmax\\nfrom review_common import (W_V5,W_V6,near_dup_groups,max_jaccard,\\n                           class_f1,macro,class1_strata)\\n\\n\\ndef dedup_train_weights(y,groups):\\n    \\"\\"\\"Accept only a training partition; never normalize using validation labels.\\"\\"\\"\\n    y=np.asarray(y);_,inv=np.unique(groups,return_inverse=True)\\n    sizes=np.bincount(inv)[inv].astype(float)\\n    w=1/np.sqrt(sizes)\\n    for c in np.unique(y):\\n        m=y==c;w[m]*=m.sum()/w[m].sum()\\n    return w\\n\\n\\ndef recipe_probabilities(svm,cat,lat):\\n    \\"\\"\\"Apply softmax separately to each model\'s margins, before ensembling folds.\\"\\"\\"\\n    base=.5*softmax(3*svm,axis=1)+.5*cat\\n    return {\'v5\':base,\'v6\':.7*base+.3*lat}\\n\\n\\ndef labels(probabilities,name):\\n    return (probabilities*(W_V5 if name==\'v5\' else W_V6)).argmax(1)\\n\\n\\ndef paired_cluster_bootstrap(y,pred_a,pred_b,groups,n=2000,seed=102):\\n    \\"\\"\\"Paired conditional bootstrap of fixed predictions, NOT a post-selection test.\\"\\"\\"\\n    y=np.asarray(y);a=np.asarray(pred_a);b=np.asarray(pred_b)\\n    if len(y)==0:raise ValueError(\'Cannot bootstrap an empty subset\')\\n    if not (len(y)==len(a)==len(b)==len(groups)):raise ValueError(\'Length mismatch\')\\n    _,inv=np.unique(groups,return_inverse=True);g=int(inv.max()+1)\\n    rng=np.random.default_rng(seed);d=np.empty(n);missing=0\\n    for i in range(n):\\n        w=np.bincount(rng.integers(0,g,g),minlength=g)[inv].astype(float)\\n        missing+=int((np.bincount(y,weights=w,minlength=4)==0).any())\\n        d[i]=macro(y,b,w)-macro(y,a,w)\\n    q=np.quantile(d,[.025,.05,.95,.975])\\n    return {\'observed_delta\':macro(y,b)-macro(y,a),\'mean\':float(d.mean()),\\n            \'sd\':float(d.std(ddof=1)),\'q025\':float(q[0]),\'q05\':float(q[1]),\\n            \'q95\':float(q[2]),\'q975\':float(q[3]),\'fraction_delta_gt_0\':float((d>0).mean()),\\n            \'groups\':g,\'rows\':len(y),\'resamples\':n,\'seed\':seed,\\n            \'fraction_resamples_missing_a_class\':missing/n,\\n            \'intervals\':\'q025-q975: central 95%; q05-q95: central 90%\',\\n            \'caveat\':\'Conditional on these selected/fitted predictions. Does not remove train/validation template overlap, selection bias, or training-sample uncertainty. Positive fraction is not a posterior probability or a calibrated p-value.\'}\\n\\n\\ndef group_summary(groups,y=None):\\n    _,inv=np.unique(groups,return_inverse=True);sizes=np.bincount(inv)\\n    out={\'groups\':len(sizes),\'largest\':int(sizes.max()),\\n         \'largest_fraction\':float(sizes.max()/len(inv)),\\n         \'rows_in_groups_ge2\':int((sizes[inv]>=2).sum()),\\n         \'group_size_quantiles\':np.quantile(sizes,[.5,.9,.99,1.]).tolist()}\\n    if y is not None:\\n        mass=np.bincount(4*inv+np.asarray(y),minlength=4*len(sizes)).reshape(-1,4)\\n        mixed=(mass>0).sum(1)>1\\n        out[\'mixed_label_groups\']=int(mixed.sum())\\n        out[\'rows_in_mixed_label_groups\']=int(sizes[mixed].sum())\\n        out[\'class1_rows_in_groups_ge2\']=int(((sizes[inv]>=2)&(np.asarray(y)==1)).sum())\\n    return out\\n", "handoff_v43.py": "\\"\\"\\"User-run Colab experiment: classwise lexical features or catalogue SVD.\\n\\nBaseline caches are predictions from the previously trained V11 pipeline,\\nnot pretrained external weights. New model/encoder fitting uses train only.\\n\\"\\"\\"\\nimport argparse,gc,hashlib,importlib.metadata,json,os,platform,shutil,sys,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom cloud_checks import guard,checked\\nfrom train_submit import load_data,metrics\\nfrom improved_features import prepare\\nfrom latent_v6 import LatentFeatures\\nfrom review_common import near_dup_groups,max_jaccard,class1_strata,macro\\nfrom codex_checked_common import paired_cluster_bootstrap\\n\\nWEIGHTS=np.array([1.,1.2,1.2,.9])\\nBEST_SHA=\'05bffeff984bd4927c8dd3e9bd3613aef6a7abce303e6e1dcb7cfa4a12012059\'\\nGATES=dict(min_oof_delta=.002,min_positive_folds=2,max_fold_drop=.003,\\n    min_novel_fake_delta=0.,min_dev_delta=.001,bootstrap_q05_positive=True)\\n\\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\\ndef prediction(p):return (p*WEIGHTS).argmax(1)\\ndef combine(ref,old,new,alpha):return checked(.7*ref+(.3-alpha)*old+alpha*new,len(ref))\\n\\ndef submission(data,test,p):\\n    sub=data[\'samplesubmission\'].copy();ids=sub.ID.copy()\\n    sub[\'spam_label\']=sub.ID.astype(str).map(dict(zip(test.id.astype(str),p)))\\n    assert not sub.spam_label.isna().any()\\n    sub[\'spam_label\']=sub.spam_label.astype(int)\\n    assert list(sub)==[\'ID\',\'spam_label\'] and len(sub)==3974\\n    assert sub.ID.equals(ids) and sub.ID.is_unique and sub.spam_label.isin(range(4)).all()\\n    return sub\\n\\ndef baseline_features(train,frames,folder,feature_id,status):\\n    \\"\\"\\"Cache only features fitted on this partition, never development labels.\\"\\"\\"\\n    path=folder/\'base_features.npz\'\\n    if path.exists():\\n        q=np.load(path,allow_pickle=False);assert q[\'feature_id\'].item()==feature_id\\n        q.close();return path,feature_id\\n    status(\'fit_review_svd128\')\\n    f=LatentFeatures();f.svd.set_params(n_components=128,random_state=102)\\n    all_x=[f.transform(train,fit=True)]+[f.transform(d) for d in frames]\\n    columns=[f\'f_{i}\' for i in range(575)];payload=dict(feature_id=feature_id,numeric_columns=np.array(columns))\\n    for prefix,x,df in zip([\'train\']+[f\'frame{i}\' for i in range(len(frames))],all_x,[train]+frames):\\n        assert len(x)==len(df) and all(c in x for c in columns)\\n        assert np.array_equal(x[[\'category\',\'rating\']].to_numpy(dtype=str),df[[\'category\',\'rating\']].astype(str).to_numpy())\\n        payload[prefix+\'_numeric\']=x[columns].to_numpy(dtype=np.float32)\\n        payload[prefix+\'_categories\']=x[[\'category\',\'rating\']].to_numpy(dtype=str)\\n    tmp=path.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,**payload);tmp.replace(path)\\n    del f,all_x,payload;gc.collect();return path,feature_id\\n\\ndef main(args):\\n    guard();start=time.time();inputs=Path(args.inputs);out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\\n    if args.trial==\'lexical\':\\n        from lexical_chi2_v41 import fit_predict,CONFIG as backend_config\\n        alpha=.15\\n    else:\\n        from catalogue_svd_v40 import fit_predict,CONFIG as backend_config\\n        alpha=.3\\n    expected=json.loads((inputs/\'inputs_manifest.json\').read_text())[\'files\']\\n    for name,h in expected.items():assert digest(inputs/name)==h,\'Input fingerprint mismatch: \'+name\\n    versions={k:importlib.metadata.version(k) for k in [\'numpy\',\'scipy\',\'pandas\',\'scikit-learn\',\'catboost\']}\\n    historical=json.loads((inputs/\'baseline/latent_oof_manifest.json\').read_text())\\n    assert versions==historical[\'versions\'],\'Use the package versions in notebook setup before running.\'\\n    sources={p.name:digest(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}\\n    manifest=dict(format=1,trial=args.trial,backend=backend_config,alpha=alpha,gates=GATES,\\n        data_and_baseline_sha256=expected,source_sha256=sources,versions=versions,seed=102)\\n    mid=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n    mp=out/\'run_manifest.json\'\\n    if mp.exists():assert json.loads(mp.read_text())==manifest,\'This output belongs to another run; choose a new output folder.\'\\n    mp.write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\\n    report=dict(trial=args.trial,pretrained=False,external_data=False,execution=\'hosted Colab CPU\',\\n        fit_data=\'train only; no dev/test feature, selector, model or calibration fitting\',\\n        baseline_user_reported_kaggle=.77436,previous_v27_user_reported_kaggle=.77229,\\n        kaggle_candidate_score=None,submission_created=False,manifest_id=mid,\\n        caveat=\'Repeated model selection on these OOF folds/dev. Bootstrap is conditional and does not remove selection bias or template overlap. No guaranteed leaderboard improvement.\')\\n    def status(stage,**kwargs):\\n        report.update(stage=stage,elapsed_seconds=time.time()-start,**kwargs)\\n        (out/\'metrics_v43.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        print(stage,json.dumps(kwargs,ensure_ascii=False),flush=True)\\n    try:\\n        (out/\'environment.json\').write_text(json.dumps(dict(python=platform.python_version(),\\n            platform=platform.system(),hosted_colab=True,cpu=os.cpu_count(),versions=versions),indent=2))\\n        status(\'validate_inputs\')\\n        data=load_data(inputs/\'data\');train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        assert [len(x) for x in [train,dev,test]]==[14306,1590,3974]\\n        ref=np.load(inputs/\'baseline/reference_v3.npz\',allow_pickle=False)\\n        lat=np.load(inputs/\'baseline/latent_oof.npz\',allow_pickle=False)\\n        full=np.load(inputs/\'baseline/latent_full.npz\',allow_pickle=False)\\n        fm=json.loads((inputs/\'baseline/latent_full_manifest.json\').read_text())\\n        assert full[\'manifest_id\'].item()==hashlib.sha256(json.dumps(fm,sort_keys=True).encode()).hexdigest()\\n        y=train.spam_label.to_numpy();folds=lat[\'folds\'];assert np.array_equal(y,lat[\'y\'])\\n        assert np.array_equal(y,ref[\'y\']) and np.array_equal(folds,ref[\'folds\'])\\n        base=checked(.7*ref[\'oof\']+.3*lat[\'candidate\'],len(y));old=prediction(base)\\n        bd=checked(.7*ref[\'dev\']+.3*full[\'dev\'],len(dev));bt=checked(.7*ref[\'test\']+.3*full[\'test\'],len(test))\\n        best=submission(data,test,prediction(bt));best_bytes=best.to_csv(index=False).encode()\\n        assert hashlib.sha256(best_bytes).hexdigest()==BEST_SHA\\n        assert digest(inputs/\'baseline/submission_best_v11.csv\')==BEST_SHA\\n        assert abs(macro(y,old)-.7749451430616494)<1e-12\\n        (out/\'submission_baseline_v11.csv\').write_bytes(best_bytes)\\n        assert np.array_equal(prediction(combine(ref[\'oof\'],lat[\'candidate\'],lat[\'candidate\'],alpha)),old)\\n        status(\'baseline_verified\',baseline_oof=metrics(y,old),baseline_dev=metrics(dev.spam_label,prediction(bd)),\\n            baseline_csv_sha256=BEST_SHA)\\n        splitter=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=102)\\n        oof=np.zeros_like(base);fold_deltas=[]\\n        for fold,(tr,va) in enumerate(splitter.split(train,y,train.comment)):\\n            assert np.all(folds[va]==fold) and not set(train.comment.iloc[tr])&set(train.comment.iloc[va])\\n            folder=out/f\'fold{fold}\';folder.mkdir(exist_ok=True)\\n            ck=folder/\'prediction.npz\';status(\'fold_start\',active_fold=fold+1,fold_deltas=fold_deltas)\\n            if ck.exists():\\n                q=np.load(ck,allow_pickle=False)\\n                assert q[\'manifest_id\'].item()==mid and np.array_equal(q[\'tr\'],tr) and np.array_equal(q[\'va\'],va)\\n                p=checked(q[\'prediction\'],len(va))\\n            else:\\n                fit=train.iloc[tr];frames=[train.iloc[va].drop(columns=\'spam_label\')]\\n                # A matching previous V23 cache is safe to reuse only under the same data/versions/fold.\\n                prior=Path(\'/content/ds102_v7_bundle/output_v23_experiment\')/f\'fold{fold}_artifacts/description_features.npz\'\\n                prior_id=hashlib.sha256(json.dumps(dict(data=historical[\'baseline_manifest\'][\'sha256\'],\\n                    dimensions=128,seed=102,description_dimensions=32,description_char_cap=4000,\\n                    tr=tr.tolist(),va=va.tolist()),sort_keys=True).encode()).hexdigest()\\n                if prior.exists():\\n                    q=np.load(prior,allow_pickle=False);assert q[\'feature_id\'].item()==prior_id\\n                    path,feature_id=prior,prior_id;q.close();status(\'reuse_matching_historical_features\',active_fold=fold+1)\\n                else:\\n                    feature_id=hashlib.sha256(json.dumps(dict(manifest_id=mid,tr=tr.tolist(),va=va.tolist()),sort_keys=True).encode()).hexdigest()\\n                    path,feature_id=baseline_features(fit,frames,folder,feature_id,status)\\n                p=fit_predict(fit,frames,folder,feature_id,status,path,feature_id)[0]\\n                tmp=ck.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,manifest_id=mid,tr=tr,va=va,prediction=p);tmp.replace(ck)\\n            oof[va]=p;new=prediction(combine(ref[\'oof\'][va],lat[\'candidate\'][va],p,alpha))\\n            fold_deltas.append(macro(y[va],new)-macro(y[va],old[va]))\\n            status(\'fold_complete\',active_fold=fold+1,fold_deltas=fold_deltas)\\n        new=prediction(combine(ref[\'oof\'],lat[\'candidate\'],oof,alpha))\\n        status(\'cv_diagnostics\',active_fold=None)\\n        template=np.zeros(len(y),bool)\\n        for fold in range(3):\\n            va=np.flatnonzero(folds==fold);tr=np.flatnonzero((folds!=fold)&(y==1))\\n            template[va]=max_jaccard(train.comment.iloc[va].tolist(),train.comment.iloc[tr].tolist())>=.6\\n        novel_delta=class1_strata(y,new,template)[\'novel\'][\'f1\']-class1_strata(y,old,template)[\'novel\'][\'f1\']\\n        delta=macro(y,new)-macro(y,old)\\n        boot=paired_cluster_bootstrap(y,old,new,near_dup_groups(train.comment.tolist(),thr=.7),n=2000)\\n        gates=dict(oof=bool(delta>=GATES[\'min_oof_delta\']),\\n            folds=bool(sum(d>0 for d in fold_deltas)>=GATES[\'min_positive_folds\']),\\n            worst_fold=bool(min(fold_deltas)>=-GATES[\'max_fold_drop\']),\\n            novel_fake=bool(novel_delta>=GATES[\'min_novel_fake_delta\']),\\n            bootstrap=bool(boot[\'q05\']>0) if GATES[\'bootstrap_q05_positive\'] else True)\\n        status(\'cv_complete\',candidate_oof=metrics(y,new),delta=delta,fold_deltas=fold_deltas,\\n            novel_fake_f1_delta=novel_delta,bootstrap=boot,gates=gates)\\n        if not all(gates.values()):status(\'rejected_on_cv\');return report\\n        folder=out/\'full\';folder.mkdir(exist_ok=True);ck=folder/\'prediction.npz\'\\n        status(\'fit_full_train\')\\n        if ck.exists():\\n            q=np.load(ck,allow_pickle=False);assert q[\'manifest_id\'].item()==mid\\n            ps=[checked(q[\'dev\'],len(dev)),checked(q[\'test\'],len(test))]\\n        else:\\n            frames=[dev.drop(columns=\'spam_label\'),test]\\n            feature_id=hashlib.sha256((mid+\':full\').encode()).hexdigest()\\n            path,feature_id=baseline_features(train,frames,folder,feature_id,status)\\n            ps=fit_predict(train,frames,folder,feature_id,status,path,feature_id)\\n            tmp=ck.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,manifest_id=mid,dev=ps[0],test=ps[1]);tmp.replace(ck)\\n        dp=prediction(combine(ref[\'dev\'],full[\'dev\'],ps[0],alpha))\\n        dev_delta=macro(dev.spam_label,dp)-macro(dev.spam_label,prediction(bd))\\n        unseen=~dev.comment.isin(set(train.comment)).to_numpy()\\n        status(\'dev_evaluated\',candidate_dev=metrics(dev.spam_label,dp),dev_delta=dev_delta,\\n            candidate_dev_without_exact_overlap=metrics(dev.spam_label.to_numpy()[unseen],dp[unseen]),\\n            baseline_dev_without_exact_overlap=metrics(dev.spam_label.to_numpy()[unseen],prediction(bd)[unseen]))\\n        if dev_delta<GATES[\'min_dev_delta\']:status(\'rejected_on_dev\');return report\\n        tp=prediction(combine(ref[\'test\'],full[\'test\'],ps[1],alpha));sub=submission(data,test,tp)\\n        path=out/\'submission_v43_candidate.csv\';sub.to_csv(path,index=False)\\n        pd.testing.assert_frame_equal(pd.read_csv(path),sub,check_dtype=False)\\n        status(\'candidate_ready_not_submitted\',submission_created=True,rows_verified=len(sub),\\n            candidate_sha256=digest(path),test_changes_vs_best=int((tp!=prediction(bt)).sum()))\\n        return report\\n    except BaseException as exc:\\n        (out/\'error.txt\').write_text(traceback.format_exc())\\n        status(\'error\',error=str(exc));raise\\n\\nif __name__==\'__main__\':\\n    parser=argparse.ArgumentParser();parser.add_argument(\'--inputs\',required=True)\\n    parser.add_argument(\'--output\',required=True);parser.add_argument(\'--trial\',choices=[\'lexical\',\'catalogue\'],default=\'lexical\')\\n    main(parser.parse_args())\\n", "experiment_v48.py": "\\"\\"\\"Train-only chi2 column weighting before SVD128; fitting labels stay inside each fold.\\n\\nBaseline caches are predictions from the previously trained V11 pipeline,\\nnot pretrained external weights. New model/encoder fitting uses train only.\\n\\"\\"\\"\\nimport argparse,gc,hashlib,importlib.metadata,json,os,platform,shutil,sys,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom cloud_checks import guard,checked\\nfrom train_submit import load_data,metrics\\nfrom improved_features import prepare\\nfrom latent_v6 import LatentFeatures\\nfrom review_common import near_dup_groups,max_jaccard,class1_strata,macro\\nfrom codex_checked_common import paired_cluster_bootstrap\\n\\nWEIGHTS=np.array([1.,1.2,1.2,.9])\\nBEST_SHA=\'05bffeff984bd4927c8dd3e9bd3613aef6a7abce303e6e1dcb7cfa4a12012059\'\\nGATES=dict(min_oof_delta=.002,min_positive_folds=2,max_fold_drop=.003,\\n    min_novel_fake_delta=0.,min_dev_delta=.001,bootstrap_q05_positive=True)\\n\\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\\ndef prediction(p):return (p*WEIGHTS).argmax(1)\\ndef combine(ref,old,new,alpha):return checked(.7*ref+(.3-alpha)*old+alpha*new,len(ref))\\n\\ndef submission(data,test,p):\\n    sub=data[\'samplesubmission\'].copy();ids=sub.ID.copy()\\n    sub[\'spam_label\']=sub.ID.astype(str).map(dict(zip(test.id.astype(str),p)))\\n    assert not sub.spam_label.isna().any()\\n    sub[\'spam_label\']=sub.spam_label.astype(int)\\n    assert list(sub)==[\'ID\',\'spam_label\'] and len(sub)==3974\\n    assert sub.ID.equals(ids) and sub.ID.is_unique and sub.spam_label.isin(range(4)).all()\\n    return sub\\n\\ndef baseline_features(train,frames,folder,feature_id,status):\\n    \\"\\"\\"Cache only features fitted on this partition, never development labels.\\"\\"\\"\\n    path=folder/\'base_features.npz\'\\n    if path.exists():\\n        q=np.load(path,allow_pickle=False);assert q[\'feature_id\'].item()==feature_id\\n        q.close();return path,feature_id\\n    status(\'fit_review_svd128\')\\n    f=LatentFeatures();f.svd.set_params(n_components=128,random_state=102)\\n    all_x=[f.transform(train,fit=True)]+[f.transform(d) for d in frames]\\n    columns=[f\'f_{i}\' for i in range(575)];payload=dict(feature_id=feature_id,numeric_columns=np.array(columns))\\n    for prefix,x,df in zip([\'train\']+[f\'frame{i}\' for i in range(len(frames))],all_x,[train]+frames):\\n        assert len(x)==len(df) and all(c in x for c in columns)\\n        assert np.array_equal(x[[\'category\',\'rating\']].to_numpy(dtype=str),df[[\'category\',\'rating\']].astype(str).to_numpy())\\n        payload[prefix+\'_numeric\']=x[columns].to_numpy(dtype=np.float32)\\n        payload[prefix+\'_categories\']=x[[\'category\',\'rating\']].to_numpy(dtype=str)\\n    tmp=path.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,**payload);tmp.replace(path)\\n    del f,all_x,payload;gc.collect();return path,feature_id\\n\\ndef main(args):\\n    guard();start=time.time();inputs=Path(args.inputs);out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\\n    assert args.trial==\'lexical\'\\n    from supervised_svd_v48 import fit_predict,CONFIG as backend_config\\n    alpha=.15\\n    expected=json.loads((inputs/\'inputs_manifest.json\').read_text())[\'files\']\\n    for name,h in expected.items():assert digest(inputs/name)==h,\'Input fingerprint mismatch: \'+name\\n    versions={k:importlib.metadata.version(k) for k in [\'numpy\',\'scipy\',\'pandas\',\'scikit-learn\',\'catboost\']}\\n    historical=json.loads((inputs/\'baseline/latent_oof_manifest.json\').read_text())\\n    assert versions==historical[\'versions\'],\'Use the package versions in notebook setup before running.\'\\n    sources={p.name:digest(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}\\n    manifest=dict(format=1,trial=args.trial,backend=backend_config,alpha=alpha,gates=GATES,\\n        data_and_baseline_sha256=expected,source_sha256=sources,versions=versions,seed=102)\\n    mid=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n    mp=out/\'run_manifest.json\'\\n    if mp.exists():assert json.loads(mp.read_text())==manifest,\'This output belongs to another run; choose a new output folder.\'\\n    mp.write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\\n    report=dict(trial=args.trial,pretrained=False,external_data=False,execution=\'hosted Colab CPU\',\\n        fit_data=\'train only; no dev/test feature, selector, model or calibration fitting\',classifier=\'CatBoost800 on supervised column-weighted SVD128; partial latent replacement15%\',\\n        baseline_user_reported_kaggle=.77436,previous_v27_user_reported_kaggle=.77229,\\n        kaggle_candidate_score=None,submission_created=False,manifest_id=mid,\\n        caveat=\'Repeated model selection on these OOF folds/dev. Bootstrap is conditional and does not remove selection bias or template overlap. No guaranteed leaderboard improvement.\')\\n    def status(stage,**kwargs):\\n        report.update(stage=stage,elapsed_seconds=time.time()-start,**kwargs)\\n        (out/\'metrics_v48.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        print(stage,json.dumps(kwargs,ensure_ascii=False),flush=True)\\n    try:\\n        (out/\'environment.json\').write_text(json.dumps(dict(python=platform.python_version(),\\n            platform=platform.system(),hosted_colab=True,cpu=os.cpu_count(),versions=versions),indent=2))\\n        status(\'validate_inputs\')\\n        data=load_data(inputs/\'data\');train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        assert [len(x) for x in [train,dev,test]]==[14306,1590,3974]\\n        ref=np.load(inputs/\'baseline/reference_v3.npz\',allow_pickle=False)\\n        lat=np.load(inputs/\'baseline/latent_oof.npz\',allow_pickle=False)\\n        full=np.load(inputs/\'baseline/latent_full.npz\',allow_pickle=False)\\n        fm=json.loads((inputs/\'baseline/latent_full_manifest.json\').read_text())\\n        assert full[\'manifest_id\'].item()==hashlib.sha256(json.dumps(fm,sort_keys=True).encode()).hexdigest()\\n        y=train.spam_label.to_numpy();folds=lat[\'folds\'];assert np.array_equal(y,lat[\'y\'])\\n        assert np.array_equal(y,ref[\'y\']) and np.array_equal(folds,ref[\'folds\'])\\n        base=checked(.7*ref[\'oof\']+.3*lat[\'candidate\'],len(y));old=prediction(base)\\n        bd=checked(.7*ref[\'dev\']+.3*full[\'dev\'],len(dev));bt=checked(.7*ref[\'test\']+.3*full[\'test\'],len(test))\\n        best=submission(data,test,prediction(bt));best_bytes=best.to_csv(index=False).encode()\\n        assert hashlib.sha256(best_bytes).hexdigest()==BEST_SHA\\n        assert digest(inputs/\'baseline/submission_best_v11.csv\')==BEST_SHA\\n        assert abs(macro(y,old)-.7749451430616494)<1e-12\\n        (out/\'submission_baseline_v11.csv\').write_bytes(best_bytes)\\n        assert np.array_equal(prediction(combine(ref[\'oof\'],lat[\'candidate\'],lat[\'candidate\'],alpha)),old)\\n        status(\'baseline_verified\',baseline_oof=metrics(y,old),baseline_dev=metrics(dev.spam_label,prediction(bd)),\\n            baseline_csv_sha256=BEST_SHA)\\n        splitter=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=102)\\n        oof=np.zeros_like(base);fold_deltas=[]\\n        for fold,(tr,va) in enumerate(splitter.split(train,y,train.comment)):\\n            assert np.all(folds[va]==fold) and not set(train.comment.iloc[tr])&set(train.comment.iloc[va])\\n            folder=out/f\'fold{fold}\';folder.mkdir(exist_ok=True)\\n            ck=folder/\'prediction.npz\';status(\'fold_start\',active_fold=fold+1,fold_deltas=fold_deltas)\\n            if ck.exists():\\n                q=np.load(ck,allow_pickle=False)\\n                assert q[\'manifest_id\'].item()==mid and np.array_equal(q[\'tr\'],tr) and np.array_equal(q[\'va\'],va)\\n                p=checked(q[\'prediction\'],len(va))\\n            else:\\n                fit=train.iloc[tr];frames=[train.iloc[va].drop(columns=\'spam_label\')]\\n                feature_id=hashlib.sha256(json.dumps(dict(manifest_id=mid,tr=tr.tolist(),va=va.tolist()),sort_keys=True).encode()).hexdigest()\\n                p=fit_predict(fit,frames,folder,feature_id,status)[0]\\n                tmp=ck.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,manifest_id=mid,tr=tr,va=va,prediction=p);tmp.replace(ck)\\n            oof[va]=p;new=prediction(combine(ref[\'oof\'][va],lat[\'candidate\'][va],p,alpha))\\n            fold_deltas.append(macro(y[va],new)-macro(y[va],old[va]))\\n            status(\'fold_complete\',active_fold=fold+1,fold_deltas=fold_deltas)\\n            remaining=2-fold\\n            if remaining>0 and (sum(d>0 for d in fold_deltas)+remaining<GATES[\'min_positive_folds\'] or min(fold_deltas)<-GATES[\'max_fold_drop\']):\\n                status(\'pruned_on_predeclared_fold_gates\',full_oof_available=False,\\n                    reason=\'Original positive-fold count or worst-fold floor is already impossible.\')\\n                return report\\n\\n        new=prediction(combine(ref[\'oof\'],lat[\'candidate\'],oof,alpha))\\n        status(\'cv_diagnostics\',active_fold=None)\\n        template=np.zeros(len(y),bool)\\n        for fold in range(3):\\n            va=np.flatnonzero(folds==fold);tr=np.flatnonzero((folds!=fold)&(y==1))\\n            template[va]=max_jaccard(train.comment.iloc[va].tolist(),train.comment.iloc[tr].tolist())>=.6\\n        novel_delta=class1_strata(y,new,template)[\'novel\'][\'f1\']-class1_strata(y,old,template)[\'novel\'][\'f1\']\\n        delta=macro(y,new)-macro(y,old)\\n        boot=paired_cluster_bootstrap(y,old,new,near_dup_groups(train.comment.tolist(),thr=.7),n=2000)\\n        gates=dict(oof=bool(delta>=GATES[\'min_oof_delta\']),\\n            folds=bool(sum(d>0 for d in fold_deltas)>=GATES[\'min_positive_folds\']),\\n            worst_fold=bool(min(fold_deltas)>=-GATES[\'max_fold_drop\']),\\n            novel_fake=bool(novel_delta>=GATES[\'min_novel_fake_delta\']),\\n            bootstrap=bool(boot[\'q05\']>0) if GATES[\'bootstrap_q05_positive\'] else True)\\n        status(\'cv_complete\',candidate_oof=metrics(y,new),delta=delta,fold_deltas=fold_deltas,\\n            novel_fake_f1_delta=novel_delta,bootstrap=boot,gates=gates)\\n        if not all(gates.values()):status(\'rejected_on_cv\');return report\\n        folder=out/\'full\';folder.mkdir(exist_ok=True);ck=folder/\'prediction.npz\'\\n        status(\'fit_full_train\')\\n        if ck.exists():\\n            q=np.load(ck,allow_pickle=False);assert q[\'manifest_id\'].item()==mid\\n            ps=[checked(q[\'dev\'],len(dev)),checked(q[\'test\'],len(test))]\\n        else:\\n            frames=[dev.drop(columns=\'spam_label\'),test]\\n            feature_id=hashlib.sha256((mid+\':full\').encode()).hexdigest()\\n            ps=fit_predict(train,frames,folder,feature_id,status)\\n            tmp=ck.with_suffix(\'.tmp.npz\');np.savez_compressed(tmp,manifest_id=mid,dev=ps[0],test=ps[1]);tmp.replace(ck)\\n        dp=prediction(combine(ref[\'dev\'],full[\'dev\'],ps[0],alpha))\\n        dev_delta=macro(dev.spam_label,dp)-macro(dev.spam_label,prediction(bd))\\n        unseen=~dev.comment.isin(set(train.comment)).to_numpy()\\n        status(\'dev_evaluated\',candidate_dev=metrics(dev.spam_label,dp),dev_delta=dev_delta,\\n            candidate_dev_without_exact_overlap=metrics(dev.spam_label.to_numpy()[unseen],dp[unseen]),\\n            baseline_dev_without_exact_overlap=metrics(dev.spam_label.to_numpy()[unseen],prediction(bd)[unseen]))\\n        if dev_delta<GATES[\'min_dev_delta\']:status(\'rejected_on_dev\');return report\\n        tp=prediction(combine(ref[\'test\'],full[\'test\'],ps[1],alpha));sub=submission(data,test,tp)\\n        path=out/\'submission_v48_candidate.csv\';sub.to_csv(path,index=False)\\n        pd.testing.assert_frame_equal(pd.read_csv(path),sub,check_dtype=False)\\n        status(\'candidate_ready_not_submitted\',submission_created=True,rows_verified=len(sub),\\n            candidate_sha256=digest(path),test_changes_vs_best=int((tp!=prediction(bt)).sum()))\\n        return report\\n    except BaseException as exc:\\n        (out/\'error.txt\').write_text(traceback.format_exc())\\n        status(\'error\',error=str(exc));raise\\n\\nif __name__==\'__main__\':\\n    parser=argparse.ArgumentParser();parser.add_argument(\'--inputs\',required=True)\\n    parser.add_argument(\'--output\',required=True);parser.add_argument(\'--trial\',choices=[\'lexical\'],default=\'lexical\')\\n    main(parser.parse_args())\\n", "supervised_svd_v48.py": "\\"\\"\\"Training-label-aware column weighting before SVD; all parameters fit in train.\\"\\"\\"\\nimport gc,json\\nfrom pathlib import Path\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy import sparse\\nfrom sklearn.decomposition import TruncatedSVD\\nfrom sklearn.feature_selection import chi2\\nfrom sklearn.preprocessing import normalize\\nfrom improved_features import FeaturesV2,dense_features\\nfrom latent_v6 import make_model\\nfrom cloud_checks import guard,checked\\n\\nCONFIG=dict(dimensions=128,svd_iterations=5,column_weight=\'sqrt chi2 / positive median\',\\n    weight_clip=[.5,4.],chi2_input=\'binary term presence\',trees=800,depth=5,\\n    learning_rate=.06,l2=8,seed=102,new_latent_coefficient=.15)\\n\\nclass SupervisedLatentFeatures:\\n    def __init__(self):\\n        self.base=FeaturesV2();self.svd=TruncatedSVD(n_components=128,n_iter=5,random_state=102)\\n    def fit(self,train):\\n        guard();x=sparse.hstack([self.base.word.fit_transform(train.comment),\\n            self.base.char.fit_transform(train.comment)],format=\'csr\')\\n        presence=x.copy();presence.data[:]=1.\\n        scores=chi2(presence,train.spam_label.to_numpy())[0]\\n        assert np.isfinite(scores).all() and (scores>=0).all()\\n        med=float(np.median(scores[scores>0]));assert med>0\\n        self.weights=np.clip(np.sqrt(scores/med),.5,4.).astype(np.float32)\\n        self.svd.fit(normalize(x.multiply(self.weights).tocsr()))\\n        self.weight_summary=dict(median_chi2=med,minimum=float(self.weights.min()),\\n            maximum=float(self.weights.max()),fraction_upper_clip=float((self.weights==4).mean()))\\n        return self\\n    def vectorize(self,text):\\n        x=sparse.hstack([self.base.word.transform(text),self.base.char.transform(text)],format=\'csr\')\\n        return normalize(x.multiply(self.weights).tocsr())\\n    def transform(self,frame):\\n        r=self.vectorize(frame.comment);t=self.vectorize(frame.product_name)\\n        z=self.svd.transform(r);p=self.svd.transform(t)\\n        cosine=np.asarray(r.multiply(t).sum(axis=1)).ravel()\\n        a=np.hstack([z,p,z*p,np.abs(z-p),cosine[:,None],dense_features(frame)]).astype(np.float32)\\n        assert a.shape==(len(frame),575) and np.isfinite(a).all()\\n        x=pd.DataFrame(a,columns=[f\'f_{i}\' for i in range(575)])\\n        x[\'category\']=frame.category.astype(str).to_numpy();x[\'rating\']=frame.rating.astype(str).to_numpy()\\n        return x\\n\\ndef fit_predict(train,frames,folder,feature_id,status,cache_path=None,cache_id=None):\\n    guard();folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\\n    status(\'fit_train_only_supervised_column_svd128\')\\n    features=SupervisedLatentFeatures().fit(train)\\n    xt=features.transform(train);external=[features.transform(df) for df in frames]\\n    status(\'fit_catboost_supervised_svd128\',feature_columns=xt.shape[1],**features.weight_summary)\\n    m=make_model();m.fit(xt,train.spam_label.to_numpy(),verbose=200)\\n    assert np.array_equal(m.classes_,[0,1,2,3])\\n    ps=[checked(m.predict_proba(x),len(df)) for x,df in zip(external,frames)]\\n    m.save_model(str(folder/\'catboost.cbm\'));joblib.dump(features,folder/\'features.joblib\',compress=3)\\n    (folder/\'metadata.json\').write_text(json.dumps(dict(config=CONFIG,feature_id=feature_id,\\n        weights=features.weight_summary),indent=2))\\n    del m,features,xt,external;gc.collect();return ps\\n", "experiment_v49.py": "\\"\\"\\"Fixed agreement committee of catalogue and supervised-column latent models.\\n\\nRule declared after V47 and first V48 fold observations: adaptive selection bias\\nremains. No meta-model is fit, no leaderboard/test labels are used. Original\\nV47/V48 rejections stay unchanged. Only0/2 may change, and only if both agree.\\n\\"\\"\\"\\nimport hashlib,importlib.metadata,json,sys,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom cloud_checks import guard,checked\\nfrom train_submit import load_data,metrics\\nfrom improved_features import prepare\\nfrom review_common import near_dup_groups,macro\\nfrom codex_checked_common import paired_cluster_bootstrap\\nfrom handoff_v43 import baseline_features,submission\\nfrom catalogue_svd_v40 import fit_predict as fit_catalogue\\nfrom supervised_svd_v48 import fit_predict as fit_supervised\\n\\nROOT=Path(\'/content/ds102_v49\');OUT=ROOT/\'run\';INPUTS=Path(\'/content/ds102_v43/inputs\')\\nW=np.array([1.,1.2,1.2,.9])\\nCONFIG=dict(rule=\'Change0/2 only when catalogue and supervised-latent predictions agree on0/2\',\\n    catalogue_alpha=.3,supervised_alpha=.15,frozen_classes=[1,3],seed=102,\\n    min_oof_gain=.002,min_positive_folds=2,max_fold_drop=.003,min_dev_gain=.001,\\n    require_bootstrap_q05_positive=True,baseline_user_reported_kaggle=.77436)\\n\\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\\ndef label(p):return (p*W).argmax(1)\\ndef decide(ref,old,cat,sup):\\n    base=label(checked(.7*ref+.3*old,len(ref)))\\n    a=label(checked(.7*ref+.3*cat,len(ref)))\\n    b=label(checked(.7*ref+.15*old+.15*sup,len(ref)))\\n    mask=np.isin(base,[0,2])&(a==b)&np.isin(a,[0,2])\\n    pred=base.copy();pred[mask]=a[mask]\\n    assert np.array_equal(pred==1,base==1) and np.array_equal(pred==3,base==3)\\n    return pred\\n\\ndef main():\\n    guard();OUT.mkdir(parents=True,exist_ok=True);start=time.time()\\n    expected=json.loads((INPUTS/\'inputs_manifest.json\').read_text())[\'files\']\\n    for n,h in expected.items():assert digest(INPUTS/n)==h\\n    historical=json.loads((INPUTS/\'baseline/latent_oof_manifest.json\').read_text())\\n    versions={k:importlib.metadata.version(k) for k in historical[\'versions\']}\\n    assert versions==historical[\'versions\']\\n    parents={47:Path(\'/content/ds102_v47/runs/catalogue_01\'),48:Path(\'/content/ds102_v48/runs/supervised_svd_01\')}\\n    parent_ids={}\\n    for v,folder in parents.items():\\n        manifest=json.loads((folder/\'run_manifest.json\').read_text())\\n        assert manifest[\'data_and_baseline_sha256\']==expected\\n        parent_ids[v]=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n    manifest=dict(config=CONFIG,versions=versions,inputs=expected,\\n        parent_manifest_sha256={str(v):digest(p/\'run_manifest.json\') for v,p in parents.items()},\\n        sources={p.name:digest(p) for p in Path(__file__).parent.glob(\'*.py\')})\\n    mid=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n    mp=OUT/\'run_manifest.json\'\\n    if mp.exists():assert json.loads(mp.read_text())==manifest\\n    mp.write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\\n    report=dict(config=CONFIG,pretrained=False,external_data=False,execution=\'hosted Colab CPU\',\\n        fit_data=\'train only; fixed vote rule, no trained meta-parameters\',submission_created=False,\\n        selection_caveat=\'Rule proposed after prior model/partial-fold outcomes; repeated OOF/dev selection remains. Paired bootstrap is conditional, not post-selection proof or guaranteed Kaggle gain.\')\\n    def status(stage,**kwargs):\\n        report.update(stage=stage,elapsed_seconds=time.time()-start,**kwargs)\\n        (OUT/\'metrics_v49.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        print(stage,json.dumps(kwargs,ensure_ascii=False),flush=True)\\n    try:\\n        data=load_data(INPUTS/\'data\');train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        ref=np.load(INPUTS/\'baseline/reference_v3.npz\');lat=np.load(INPUTS/\'baseline/latent_oof.npz\');full=np.load(INPUTS/\'baseline/latent_full.npz\')\\n        y=train.spam_label.to_numpy();folds=lat[\'folds\']\\n        assert np.array_equal(y,lat[\'y\']) and np.array_equal(y,ref[\'y\']) and np.array_equal(folds,ref[\'folds\'])\\n        full_manifest=json.loads((INPUTS/\'baseline/latent_full_manifest.json\').read_text())\\n        assert full[\'manifest_id\'].item()==hashlib.sha256(json.dumps(full_manifest,sort_keys=True).encode()).hexdigest()\\n        baseline=label(.7*ref[\'oof\']+.3*lat[\'candidate\']);pred=baseline.copy();fd=[]\\n        splitter=StratifiedGroupKFold(3,shuffle=True,random_state=102)\\n        for f,(tr,va) in enumerate(splitter.split(train,y,train.comment)):\\n            assert np.all(folds[va]==f) and not set(train.comment.iloc[tr])&set(train.comment.iloc[va])\\n            ps={};status(\'fold_start\',active_fold=f+1,fold_deltas=fd)\\n            for v in [47,48]:\\n                cache=parents[v]/f\'fold{f}/prediction.npz\'\\n                if cache.exists():\\n                    q=np.load(cache,allow_pickle=False)\\n                    assert q[\'manifest_id\'].item()==parent_ids[v] and np.array_equal(q[\'tr\'],tr) and np.array_equal(q[\'va\'],va)\\n                    ps[v]=checked(q[\'prediction\'],len(va))\\n                else:\\n                    assert v==48,\'Catalogue parent must have completed all folds\'\\n                    folder=OUT/f\'fold{f}_supervised_completion\';folder.mkdir(exist_ok=True)\\n                    cache=folder/\'prediction.npz\'\\n                    if cache.exists():\\n                        q=np.load(cache);assert q[\'manifest_id\'].item()==mid and np.array_equal(q[\'tr\'],tr) and np.array_equal(q[\'va\'],va)\\n                        ps[v]=checked(q[\'prediction\'],len(va))\\n                    else:\\n                        feature_id=hashlib.sha256((mid+\':fold\'+str(f)).encode()).hexdigest()\\n                        status(\'complete_missing_supervised_component\',active_fold=f+1)\\n                        ps[v]=fit_supervised(train.iloc[tr],[train.iloc[va].drop(columns=\'spam_label\')],folder,feature_id,status)[0]\\n                        np.savez_compressed(cache,manifest_id=mid,tr=tr,va=va,prediction=ps[v])\\n            pred[va]=decide(ref[\'oof\'][va],lat[\'candidate\'][va],ps[47],ps[48])\\n            fd.append(macro(y[va],pred[va])-macro(y[va],baseline[va]))\\n            status(\'fold_complete\',active_fold=f+1,fold_deltas=fd)\\n            if f<2 and (sum(x>0 for x in fd)+(2-f)<2 or min(fd)<-.003):\\n                status(\'pruned_on_predeclared_fold_gates\',full_oof_available=False);return\\n        assert np.array_equal(pred==1,baseline==1) and np.array_equal(pred==3,baseline==3)\\n        delta=macro(y,pred)-macro(y,baseline)\\n        status(\'cv_diagnostics\',active_fold=None,delta=delta,fold_deltas=fd)\\n        boot=paired_cluster_bootstrap(y,baseline,pred,near_dup_groups(train.comment.tolist(),thr=.7),n=2000)\\n        gates=dict(oof=bool(delta>=.002),folds=bool(sum(x>0 for x in fd)>=2),worst_fold=bool(min(fd)>=-.003),\\n            novel_fake=True,bootstrap=bool(boot[\'q05\']>0))\\n        status(\'cv_complete\',baseline_oof=metrics(y,baseline),candidate_oof=metrics(y,pred),\\n            delta=delta,fold_deltas=fd,novel_fake_f1_delta=0.,class1_and3_pred_mask_exactly_preserved=True,\\n            bootstrap=boot,gates=gates)\\n        np.savez_compressed(OUT/\'oof.npz\',y=y,folds=folds,prediction=pred)\\n        if not all(gates.values()):status(\'rejected_on_cv\');return\\n        frames=[dev.drop(columns=\'spam_label\'),test];external={}\\n        for v,fn in [(47,fit_catalogue),(48,fit_supervised)]:\\n            folder=OUT/f\'full_component{v}\';folder.mkdir(exist_ok=True);cache=folder/\'predictions.npz\'\\n            if cache.exists():\\n                q=np.load(cache);assert q[\'manifest_id\'].item()==mid\\n                external[v]=[checked(q[\'dev\'],len(dev)),checked(q[\'test\'],len(test))]\\n            else:\\n                fid=hashlib.sha256((mid+\':full\'+str(v)).encode()).hexdigest()\\n                status(\'fit_full_train\',component=v)\\n                if v==47:\\n                    path,fid=baseline_features(train,frames,folder,fid,status)\\n                    external[v]=fn(train,frames,folder,fid,status,path,fid)\\n                else:external[v]=fn(train,frames,folder,fid,status)\\n                np.savez_compressed(cache,manifest_id=mid,dev=external[v][0],test=external[v][1])\\n        old_dev=label(.7*ref[\'dev\']+.3*full[\'dev\'])\\n        dp=decide(ref[\'dev\'],full[\'dev\'],external[47][0],external[48][0])\\n        dev_delta=macro(dev.spam_label,dp)-macro(dev.spam_label,old_dev)\\n        status(\'dev_evaluated\',baseline_dev=metrics(dev.spam_label,old_dev),candidate_dev=metrics(dev.spam_label,dp),dev_delta=dev_delta)\\n        if dev_delta<.001:status(\'rejected_on_dev\');return\\n        old_test=label(.7*ref[\'test\']+.3*full[\'test\'])\\n        best=submission(data,test,old_test)\\n        assert hashlib.sha256(best.to_csv(index=False).encode()).hexdigest()==\'05bffeff984bd4927c8dd3e9bd3613aef6a7abce303e6e1dcb7cfa4a12012059\'\\n        tp=decide(ref[\'test\'],full[\'test\'],external[47][1],external[48][1]);sub=submission(data,test,tp)\\n        path=OUT/\'submission_v49_candidate.csv\';sub.to_csv(path,index=False)\\n        status(\'candidate_ready_not_submitted\',submission_created=True,rows_verified=len(sub),sha256=digest(path),test_changes_vs_best=int((tp!=old_test).sum()))\\n    except BaseException as exc:\\n        (OUT/\'error.txt\').write_text(traceback.format_exc());status(\'error\',error=str(exc));raise\\n\\nif __name__==\'__main__\':main()\\n", "experiment_v50.py": "\\"\\"\\"Fixed agreement committee of catalogue and supervised-column latent models.\\n\\nRule declared after V47/V48 outcomes and first two V49 committee fold observations: adaptive selection bias\\nremains. No meta-model is fit, no leaderboard/test labels are used. Original\\nV47/V48 rejections stay unchanged. Only class1 is frozen; other labels change only when both models agree.\\n\\"\\"\\"\\nimport hashlib,importlib.metadata,json,sys,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nfrom sklearn.model_selection import StratifiedGroupKFold\\nfrom cloud_checks import guard,checked\\nfrom train_submit import load_data,metrics\\nfrom improved_features import prepare\\nfrom review_common import near_dup_groups,macro\\nfrom codex_checked_common import paired_cluster_bootstrap\\nfrom handoff_v43 import baseline_features,submission\\nfrom catalogue_svd_v40 import fit_predict as fit_catalogue\\nfrom supervised_svd_v48 import fit_predict as fit_supervised\\n\\nROOT=Path(\'/content/ds102_v50\');OUT=ROOT/\'run\';INPUTS=Path(\'/content/ds102_v43/inputs\')\\nW=np.array([1.,1.2,1.2,.9])\\nCONFIG=dict(rule=\'Keep original class1 predictions; change0/2/3 only when both independent feature-view predictions agree on0/2/3\',\\n    catalogue_alpha=.3,supervised_alpha=.15,frozen_classes=[1],seed=102,\\n    min_oof_gain=.002,min_positive_folds=2,max_fold_drop=.003,min_dev_gain=.001,\\n    require_bootstrap_q05_positive=True,baseline_user_reported_kaggle=.77436)\\n\\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\\ndef label(p):return (p*W).argmax(1)\\ndef decide(ref,old,cat,sup):\\n    base=label(checked(.7*ref+.3*old,len(ref)))\\n    a=label(checked(.7*ref+.3*cat,len(ref)))\\n    b=label(checked(.7*ref+.15*old+.15*sup,len(ref)))\\n    mask=np.isin(base,[0,2,3])&(a==b)&np.isin(a,[0,2,3])\\n    pred=base.copy();pred[mask]=a[mask]\\n    assert np.array_equal(pred==1,base==1)\\n    return pred\\n\\ndef main():\\n    guard();OUT.mkdir(parents=True,exist_ok=True);start=time.time()\\n    expected=json.loads((INPUTS/\'inputs_manifest.json\').read_text())[\'files\']\\n    for n,h in expected.items():assert digest(INPUTS/n)==h\\n    historical=json.loads((INPUTS/\'baseline/latent_oof_manifest.json\').read_text())\\n    versions={k:importlib.metadata.version(k) for k in historical[\'versions\']}\\n    assert versions==historical[\'versions\']\\n    parents={47:Path(\'/content/ds102_v47/runs/catalogue_01\'),48:Path(\'/content/ds102_v48/runs/supervised_svd_01\')}\\n    parent_ids={}\\n    for v,folder in parents.items():\\n        manifest=json.loads((folder/\'run_manifest.json\').read_text())\\n        assert manifest[\'data_and_baseline_sha256\']==expected\\n        parent_ids[v]=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n    prior49=Path(\'/content/ds102_v49/run\');m49=json.loads((prior49/\'run_manifest.json\').read_text())\\n    assert m49[\'inputs\']==expected\\n    for backend in [\'catalogue_svd_v40.py\',\'supervised_svd_v48.py\']:\\n        assert m49[\'sources\'][backend]==digest(Path(__file__).parent/backend)\\n    mid49=hashlib.sha256(json.dumps(m49,sort_keys=True).encode()).hexdigest()\\n    manifest=dict(config=CONFIG,versions=versions,inputs=expected,\\n        parent_manifest_sha256={str(v):digest(p/\'run_manifest.json\') for v,p in parents.items()},\\n        prior49_manifest_sha256=digest(prior49/\'run_manifest.json\'),\\n        sources={p.name:digest(p) for p in Path(__file__).parent.glob(\'*.py\')})\\n    mid=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n    mp=OUT/\'run_manifest.json\'\\n    if mp.exists():assert json.loads(mp.read_text())==manifest\\n    mp.write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\\n    report=dict(config=CONFIG,pretrained=False,external_data=False,execution=\'hosted Colab CPU\',\\n        fit_data=\'train only; fixed vote rule, no trained meta-parameters\',submission_created=False,\\n        selection_caveat=\'Rule proposed after prior model and two partial V49 committee fold outcomes; repeated OOF/dev selection remains. Paired bootstrap is conditional, not post-selection proof or guaranteed Kaggle gain.\')\\n    def status(stage,**kwargs):\\n        report.update(stage=stage,elapsed_seconds=time.time()-start,**kwargs)\\n        (OUT/\'metrics_v50.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        print(stage,json.dumps(kwargs,ensure_ascii=False),flush=True)\\n    try:\\n        data=load_data(INPUTS/\'data\');train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        ref=np.load(INPUTS/\'baseline/reference_v3.npz\');lat=np.load(INPUTS/\'baseline/latent_oof.npz\');full=np.load(INPUTS/\'baseline/latent_full.npz\')\\n        y=train.spam_label.to_numpy();folds=lat[\'folds\']\\n        assert np.array_equal(y,lat[\'y\']) and np.array_equal(y,ref[\'y\']) and np.array_equal(folds,ref[\'folds\'])\\n        full_manifest=json.loads((INPUTS/\'baseline/latent_full_manifest.json\').read_text())\\n        assert full[\'manifest_id\'].item()==hashlib.sha256(json.dumps(full_manifest,sort_keys=True).encode()).hexdigest()\\n        baseline=label(.7*ref[\'oof\']+.3*lat[\'candidate\']);pred=baseline.copy();fd=[]\\n        splitter=StratifiedGroupKFold(3,shuffle=True,random_state=102)\\n        for f,(tr,va) in enumerate(splitter.split(train,y,train.comment)):\\n            assert np.all(folds[va]==f) and not set(train.comment.iloc[tr])&set(train.comment.iloc[va])\\n            ps={};status(\'fold_start\',active_fold=f+1,fold_deltas=fd)\\n            for v in [47,48]:\\n                cache=parents[v]/f\'fold{f}/prediction.npz\'\\n                if cache.exists():\\n                    q=np.load(cache,allow_pickle=False)\\n                    assert q[\'manifest_id\'].item()==parent_ids[v] and np.array_equal(q[\'tr\'],tr) and np.array_equal(q[\'va\'],va)\\n                    ps[v]=checked(q[\'prediction\'],len(va))\\n                else:\\n                    assert v==48,\'Catalogue parent must have completed all folds\'\\n                    shared=prior49/f\'fold{f}_supervised_completion/prediction.npz\'\\n                    if shared.exists():\\n                        q=np.load(shared,allow_pickle=False)\\n                        assert q[\'manifest_id\'].item()==mid49 and np.array_equal(q[\'tr\'],tr) and np.array_equal(q[\'va\'],va)\\n                        ps[v]=checked(q[\'prediction\'],len(va))\\n                        continue\\n\\n                    folder=OUT/f\'fold{f}_supervised_completion\';folder.mkdir(exist_ok=True)\\n                    cache=folder/\'prediction.npz\'\\n                    if cache.exists():\\n                        q=np.load(cache);assert q[\'manifest_id\'].item()==mid and np.array_equal(q[\'tr\'],tr) and np.array_equal(q[\'va\'],va)\\n                        ps[v]=checked(q[\'prediction\'],len(va))\\n                    else:\\n                        feature_id=hashlib.sha256((mid+\':fold\'+str(f)).encode()).hexdigest()\\n                        status(\'complete_missing_supervised_component\',active_fold=f+1)\\n                        ps[v]=fit_supervised(train.iloc[tr],[train.iloc[va].drop(columns=\'spam_label\')],folder,feature_id,status)[0]\\n                        np.savez_compressed(cache,manifest_id=mid,tr=tr,va=va,prediction=ps[v])\\n            pred[va]=decide(ref[\'oof\'][va],lat[\'candidate\'][va],ps[47],ps[48])\\n            fd.append(macro(y[va],pred[va])-macro(y[va],baseline[va]))\\n            status(\'fold_complete\',active_fold=f+1,fold_deltas=fd)\\n            if f<2 and (sum(x>0 for x in fd)+(2-f)<2 or min(fd)<-.003):\\n                status(\'pruned_on_predeclared_fold_gates\',full_oof_available=False);return\\n        assert np.array_equal(pred==1,baseline==1)\\n        delta=macro(y,pred)-macro(y,baseline)\\n        status(\'cv_diagnostics\',active_fold=None,delta=delta,fold_deltas=fd)\\n        boot=paired_cluster_bootstrap(y,baseline,pred,near_dup_groups(train.comment.tolist(),thr=.7),n=2000)\\n        gates=dict(oof=bool(delta>=.002),folds=bool(sum(x>0 for x in fd)>=2),worst_fold=bool(min(fd)>=-.003),\\n            novel_fake=True,bootstrap=bool(boot[\'q05\']>0))\\n        status(\'cv_complete\',baseline_oof=metrics(y,baseline),candidate_oof=metrics(y,pred),\\n            delta=delta,fold_deltas=fd,novel_fake_f1_delta=0.,class1_pred_mask_exactly_preserved=True,\\n            bootstrap=boot,gates=gates)\\n        np.savez_compressed(OUT/\'oof.npz\',y=y,folds=folds,prediction=pred)\\n        if not all(gates.values()):status(\'rejected_on_cv\');return\\n        frames=[dev.drop(columns=\'spam_label\'),test];external={}\\n        for v,fn in [(47,fit_catalogue),(48,fit_supervised)]:\\n            folder=OUT/f\'full_component{v}\';folder.mkdir(exist_ok=True);cache=folder/\'predictions.npz\'\\n            shared=prior49/f\'full_component{v}/predictions.npz\'\\n            if not cache.exists() and shared.exists():\\n                q=np.load(shared,allow_pickle=False);assert q[\'manifest_id\'].item()==mid49\\n                np.savez_compressed(cache,manifest_id=mid,dev=checked(q[\'dev\'],len(dev)),test=checked(q[\'test\'],len(test)))\\n            if cache.exists():\\n                q=np.load(cache);assert q[\'manifest_id\'].item()==mid\\n                external[v]=[checked(q[\'dev\'],len(dev)),checked(q[\'test\'],len(test))]\\n            else:\\n                fid=hashlib.sha256((mid+\':full\'+str(v)).encode()).hexdigest()\\n                status(\'fit_full_train\',component=v)\\n                if v==47:\\n                    path,fid=baseline_features(train,frames,folder,fid,status)\\n                    external[v]=fn(train,frames,folder,fid,status,path,fid)\\n                else:external[v]=fn(train,frames,folder,fid,status)\\n                np.savez_compressed(cache,manifest_id=mid,dev=external[v][0],test=external[v][1])\\n        old_dev=label(.7*ref[\'dev\']+.3*full[\'dev\'])\\n        dp=decide(ref[\'dev\'],full[\'dev\'],external[47][0],external[48][0])\\n        dev_delta=macro(dev.spam_label,dp)-macro(dev.spam_label,old_dev)\\n        status(\'dev_evaluated\',baseline_dev=metrics(dev.spam_label,old_dev),candidate_dev=metrics(dev.spam_label,dp),dev_delta=dev_delta)\\n        if dev_delta<.001:status(\'rejected_on_dev\');return\\n        old_test=label(.7*ref[\'test\']+.3*full[\'test\'])\\n        best=submission(data,test,old_test)\\n        assert hashlib.sha256(best.to_csv(index=False).encode()).hexdigest()==\'05bffeff984bd4927c8dd3e9bd3613aef6a7abce303e6e1dcb7cfa4a12012059\'\\n        tp=decide(ref[\'test\'],full[\'test\'],external[47][1],external[48][1]);sub=submission(data,test,tp)\\n        path=OUT/\'submission_v50_candidate.csv\';sub.to_csv(path,index=False)\\n        status(\'candidate_ready_not_submitted\',submission_created=True,rows_verified=len(sub),sha256=digest(path),test_changes_vs_best=int((tp!=old_test).sum()))\\n    except BaseException as exc:\\n        (OUT/\'error.txt\').write_text(traceback.format_exc());status(\'error\',error=str(exc));raise\\n\\nif __name__==\'__main__\':main()\\n", "finalize_v51_experimental.py": "\\"\\"\\"Exploratory full-fit follow-up of fixed V50, with stronger dev conditions.\\n\\nV50\'s original +.002 OOF gate remains FAILED. No original report/gate is edited.\\nThis follow-up is not a strict-CV promotion. Extra dev tests were declared\\nbefore any full-fit/dev predictions of the committee were observed.\\n\\"\\"\\"\\nimport hashlib,importlib.metadata,json,time,traceback\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom cloud_checks import guard,checked\\nfrom train_submit import load_data,metrics\\nfrom improved_features import prepare\\nfrom handoff_v43 import baseline_features,submission\\nfrom experiment_v50 import decide,label\\nfrom catalogue_svd_v40 import fit_predict as fit_catalogue\\nfrom supervised_svd_v48 import fit_predict as fit_supervised\\nfrom review_common import macro,near_dup_groups\\nfrom codex_checked_common import paired_cluster_bootstrap\\n\\nROOT=Path(\'/content/ds102_v51\');OUT=ROOT/\'run\';INPUTS=Path(\'/content/ds102_v43/inputs\')\\nCONFIG=dict(qualification=\'Exploratory follow-up, original V50 strict CV gate rejected\',\\n    original_min_oof_gain=.002,observed_oof_gain=.001463981731077979,\\n    min_dev_gain=.003,min_dev_no_exact_overlap_gain=.003,require_dev_bootstrap_q05_positive=True,\\n    original_class1_predictions_frozen=True,baseline_user_reported_kaggle=.77436)\\n\\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\\n\\ndef main():\\n    guard();OUT.mkdir(parents=True,exist_ok=True);start=time.time()\\n    parent=Path(\'/content/ds102_v50/run\');cv=json.loads((parent/\'metrics_v50.json\').read_text())\\n    pm=json.loads((parent/\'run_manifest.json\').read_text())\\n    assert cv[\'stage\']==\'rejected_on_cv\' and not cv[\'gates\'][\'oof\']\\n    assert all(cv[\'gates\'][k] for k in [\'folds\',\'worst_fold\',\'novel_fake\',\'bootstrap\'])\\n    assert cv[\'delta\']>=.001 and cv[\'bootstrap\'][\'q025\']>0\\n    assert cv[\'novel_fake_f1_delta\']==0 and cv[\'class1_pred_mask_exactly_preserved\']\\n    for name in [\'catalogue_svd_v40.py\',\'supervised_svd_v48.py\',\'experiment_v50.py\',\'handoff_v43.py\']:\\n        assert digest(Path(__file__).parent/name)==pm[\'sources\'][name],name\\n    expected=json.loads((INPUTS/\'inputs_manifest.json\').read_text())[\'files\']\\n    assert expected==pm[\'inputs\']\\n    for n,h in expected.items():assert digest(INPUTS/n)==h,n\\n    versions={k:importlib.metadata.version(k) for k in pm[\'versions\']};assert versions==pm[\'versions\']\\n    manifest=dict(config=CONFIG,versions=versions,inputs=expected,\\n        parent_cv_sha256=digest(parent/\'metrics_v50.json\'),parent_manifest_sha256=digest(parent/\'run_manifest.json\'),\\n        sources={p.name:digest(p) for p in Path(__file__).parent.glob(\'*.py\')})\\n    mid=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()\\n    mp=OUT/\'run_manifest.json\'\\n    if mp.exists():assert json.loads(mp.read_text())==manifest,\'Use a fresh output directory for changed source\'\\n    mp.write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\\n    report=dict(config=CONFIG,cv=cv,pretrained=False,external_data=False,\\n        execution=\'hosted Colab CPU\',fit_data=\'train only; dev/test excluded from every fitting step\',\\n        original_cv_gate_passed=False,experimental=True,submission_created=False,\\n        caveat=\'OOF and dev have been reused for many experiments. V50 practical effect-size floor failed. This exploratory follow-up keeps the fixed rule and requires a larger dev gain and positive conditional dev bootstrap. Neither test removes selection bias or guarantees Kaggle improvement.\')\\n    def status(stage,**kwargs):\\n        report.update(stage=stage,elapsed_seconds=time.time()-start,**kwargs)\\n        (OUT/\'metrics_v51_experimental.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n        print(stage,json.dumps(kwargs,ensure_ascii=False),flush=True)\\n    try:\\n        data=load_data(INPUTS/\'data\');train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n        ref=np.load(INPUTS/\'baseline/reference_v3.npz\');full=np.load(INPUTS/\'baseline/latent_full.npz\')\\n        assert np.array_equal(ref[\'y\'],train.spam_label.to_numpy())\\n        fm=json.loads((INPUTS/\'baseline/latent_full_manifest.json\').read_text())\\n        assert full[\'manifest_id\'].item()==hashlib.sha256(json.dumps(fm,sort_keys=True).encode()).hexdigest()\\n        baseline_test=label(.7*ref[\'test\']+.3*full[\'test\'])\\n        best=submission(data,test,baseline_test)\\n        assert hashlib.sha256(best.to_csv(index=False).encode()).hexdigest()==\'05bffeff984bd4927c8dd3e9bd3613aef6a7abce303e6e1dcb7cfa4a12012059\'\\n        report[\'baseline_matches_best_submission\']=True\\n        frames=[dev.drop(columns=\'spam_label\'),test];external={}\\n        for v,fn in [(47,fit_catalogue),(48,fit_supervised)]:\\n            folder=OUT/f\'full_component{v}\';folder.mkdir(exist_ok=True);cache=folder/\'predictions.npz\'\\n            status(\'full_train_component_start\',component=v)\\n            if cache.exists():\\n                q=np.load(cache);assert q[\'manifest_id\'].item()==mid\\n                external[v]=[checked(q[\'dev\'],len(dev)),checked(q[\'test\'],len(test))]\\n            else:\\n                fid=hashlib.sha256((mid+\':full\'+str(v)).encode()).hexdigest()\\n                if v==47:\\n                    path,fid=baseline_features(train,frames,folder,fid,status)\\n                    external[v]=fn(train,frames,folder,fid,status,path,fid)\\n                else:external[v]=fn(train,frames,folder,fid,status)\\n                temp=cache.with_suffix(\'.tmp.npz\');np.savez_compressed(temp,manifest_id=mid,dev=external[v][0],test=external[v][1]);temp.replace(cache)\\n        old=label(.7*ref[\'dev\']+.3*full[\'dev\'])\\n        dp=decide(ref[\'dev\'],full[\'dev\'],external[47][0],external[48][0])\\n        assert np.array_equal(old==1,dp==1)\\n        y=dev.spam_label.to_numpy();unseen=~dev.comment.isin(set(train.comment)).to_numpy()\\n        gain=macro(y,dp)-macro(y,old);gain_unseen=macro(y[unseen],dp[unseen])-macro(y[unseen],old[unseen])\\n        status(\'dev_diagnostics\',dev_delta=gain,dev_no_exact_overlap_delta=gain_unseen)\\n        boot=paired_cluster_bootstrap(y,old,dp,near_dup_groups(dev.comment.tolist(),thr=.7),n=5000)\\n        gates=dict(dev_gain=bool(gain>=.003),dev_no_exact_overlap_gain=bool(gain_unseen>=.003),\\n            conditional_dev_bootstrap=bool(boot[\'q05\']>0),original_fake_decisions_preserved=bool(np.array_equal(old==1,dp==1)))\\n        status(\'dev_evaluated\',baseline_dev=metrics(y,old),candidate_dev=metrics(y,dp),\\n            baseline_dev_without_exact_overlap=metrics(y[unseen],old[unseen]),\\n            candidate_dev_without_exact_overlap=metrics(y[unseen],dp[unseen]),dev_delta=gain,\\n            dev_no_exact_overlap_delta=gain_unseen,dev_bootstrap=boot,additional_dev_gates=gates)\\n        if not all(gates.values()):status(\'exploratory_followup_rejected_on_dev\');return\\n        tp=decide(ref[\'test\'],full[\'test\'],external[47][1],external[48][1]);changes=int((tp!=baseline_test).sum())\\n        assert np.array_equal(tp==1,baseline_test==1)\\n        if changes==0:status(\'no_test_changes_no_candidate_export\');return\\n        sub=submission(data,test,tp);path=OUT/\'submission_v51_experimental.csv\';sub.to_csv(path,index=False)\\n        pd.testing.assert_frame_equal(pd.read_csv(path),sub,check_dtype=False)\\n        status(\'experimental_candidate_ready_not_submitted\',submission_created=True,rows_verified=len(sub),\\n            candidate_sha256=digest(path),test_changes_vs_best=changes)\\n    except BaseException as exc:\\n        (OUT/\'error.txt\').write_text(traceback.format_exc());status(\'error\',error=str(exc));raise\\n\\nif __name__==\'__main__\':main()\\n", "export_v52_exploratory.py": "\\"\\"\\"Export an explicitly exploratory V50-rule CSV from actual V51 full fits.\\n\\nV50 strict-CV and V51 extra-dev rejections remain unchanged. Export is a\\nreviewable experiment requested by the user, NOT a promotion under those gates.\\nPoint estimates improved on OOF/dev; dev uncertainty still includes declines.\\nNo new parameters are selected or fit here unless replay needs missing full fits.\\n\\"\\"\\"\\nimport hashlib,json,time\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom cloud_checks import guard,checked\\nfrom train_submit import load_data,metrics\\nfrom improved_features import prepare\\nfrom handoff_v43 import submission\\nfrom experiment_v50 import decide,label\\n\\nINPUTS=Path(\'/content/ds102_v43/inputs\');FULL=Path(\'/content/ds102_v51/run\')\\nOUT=Path(\'/content/ds102_v52_exploratory\')\\n\\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\\n\\ndef main():\\n    guard();start=time.time();OUT.mkdir(exist_ok=True)\\n    required=[FULL/\'metrics_v51_experimental.json\',FULL/\'full_component47/predictions.npz\',FULL/\'full_component48/predictions.npz\']\\n    if not all(p.exists() for p in required):\\n        # Clean replay fits the exact frozen final recipe; training/dev policies\\n        # and the original rejection evidence are not rewritten by this export.\\n        from finalize_v51_experimental import main as fit_final\\n        fit_final()\\n    assert all(p.exists() for p in required)\\n    rp=FULL/\'metrics_v51_experimental.json\';before=digest(rp);r=json.loads(rp.read_text());cv=r[\'cv\']\\n    assert r[\'experimental\'] and not r[\'original_cv_gate_passed\']\\n    assert cv[\'stage\']==\'rejected_on_cv\' and cv[\'delta\']>0 and cv[\'bootstrap\'][\'q025\']>0\\n    assert cv[\'class1_pred_mask_exactly_preserved\'] and cv[\'novel_fake_f1_delta\']==0\\n    assert r[\'dev_delta\']>0 and r[\'dev_no_exact_overlap_delta\']>0\\n    manifest51=json.loads((FULL/\'run_manifest.json\').read_text())\\n    mid51=hashlib.sha256(json.dumps(manifest51,sort_keys=True).encode()).hexdigest()\\n    expected=json.loads((INPUTS/\'inputs_manifest.json\').read_text())[\'files\']\\n    assert expected==manifest51[\'inputs\']\\n    for n,h in expected.items():assert digest(INPUTS/n)==h,n\\n    assert digest(Path(__file__).parent/\'experiment_v50.py\')==manifest51[\'sources\'][\'experiment_v50.py\']\\n    data=load_data(INPUTS/\'data\');train,dev,test=[prepare(data[k]) for k in [\'train\',\'dev\',\'test\']]\\n    ref=np.load(INPUTS/\'baseline/reference_v3.npz\');full=np.load(INPUTS/\'baseline/latent_full.npz\')\\n    assert np.array_equal(ref[\'y\'],train.spam_label.to_numpy())\\n    fm=json.loads((INPUTS/\'baseline/latent_full_manifest.json\').read_text())\\n    assert full[\'manifest_id\'].item()==hashlib.sha256(json.dumps(fm,sort_keys=True).encode()).hexdigest()\\n    ps={}\\n    for v in [47,48]:\\n        q=np.load(FULL/f\'full_component{v}/predictions.npz\',allow_pickle=False)\\n        assert q[\'manifest_id\'].item()==mid51\\n        ps[v]=[checked(q[\'dev\'],len(dev)),checked(q[\'test\'],len(test))]\\n    old_dev=label(.7*ref[\'dev\']+.3*full[\'dev\']);old_test=label(.7*ref[\'test\']+.3*full[\'test\'])\\n    dp=decide(ref[\'dev\'],full[\'dev\'],ps[47][0],ps[48][0]);tp=decide(ref[\'test\'],full[\'test\'],ps[47][1],ps[48][1])\\n    assert np.array_equal(dp==1,old_dev==1) and np.array_equal(tp==1,old_test==1)\\n    dev_metrics=metrics(dev.spam_label.to_numpy(),dp)\\n    assert abs(dev_metrics[\'macro_f1\']-r[\'candidate_dev\'][\'macro_f1\'])<1e-12\\n    best=submission(data,test,old_test)\\n    assert hashlib.sha256(best.to_csv(index=False).encode()).hexdigest()==\'05bffeff984bd4927c8dd3e9bd3613aef6a7abce303e6e1dcb7cfa4a12012059\'\\n    changes=int((tp!=old_test).sum());assert changes>0,\'Same labels as best file; no new candidate\'\\n    sub=submission(data,test,tp);path=OUT/\'submission_v52_exploratory.csv\';sub.to_csv(path,index=False)\\n    pd.testing.assert_frame_equal(pd.read_csv(path),sub,check_dtype=False)\\n    assert digest(rp)==before,\'Original rejection report must remain unchanged\'\\n    report=dict(stage=\'exploratory_candidate_ready_not_submitted\',experimental=True,\\n        recommendation=\'Exploratory test only; NOT a strict-CV promotion or confirmed Kaggle improvement\',\\n        original_cv_gate_passed=False,additional_dev_gates_passed=all(r[\'additional_dev_gates\'].values()),\\n        original_rejections_preserved=True,original_followup_report_sha256=before,\\n        baseline_user_reported_kaggle=.77436,candidate_kaggle_score=None,\\n        cv=cv,baseline_dev=r[\'baseline_dev\'],candidate_dev=dev_metrics,dev_delta=r[\'dev_delta\'],\\n        dev_no_exact_overlap_delta=r[\'dev_no_exact_overlap_delta\'],dev_bootstrap=r[\'dev_bootstrap\'],\\n        original_additional_dev_gates=r[\'additional_dev_gates\'],\\n        rows_verified=len(sub),test_changes_vs_best=changes,dev_changes_vs_best=int((dp!=old_dev).sum()),\\n        original_fake_review_prediction_mask_preserved=True,\\n        prediction_counts={str(c):int((tp==c).sum()) for c in range(4)},\\n        candidate_sha256=digest(path),pretrained=False,external_data=False,\\n        fit_data=\'train only; dev/test excluded from all fitting\',execution=\'hosted Colab CPU\',\\n        recipe=\'Fixed agreement of catalogueSVD32+SVD128 CatBoost and supervised-column-SVD128 CatBoost; base remainsV11. Only change non-class1 predictions if both fixed model views agree on0/2/3.\',\\n        caveat=\'The original .002 OOF floor and extra .003 dev/bootstrap conditions were not met. OOF/dev have been repeatedly reused. Both point estimates increased, but dev interval includes decreases. Do not claim this file will improve Kaggle. No test labels or external model weights used.\',\\n        elapsed_export_seconds=time.time()-start)\\n    (OUT/\'metrics_v52_exploratory.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2))\\n    manifest=dict(source_sha256={p.name:digest(p) for p in Path(__file__).parent.glob(\'*.py\')},\\n        inputs=expected,full_manifest_sha256=digest(FULL/\'run_manifest.json\'),\\n        full_prediction_sha256={str(v):digest(FULL/f\'full_component{v}/predictions.npz\') for v in [47,48]},\\n        original_cv_report_sha256=digest(Path(\'/content/ds102_v50/run/metrics_v50.json\')),\\n        original_dev_rejection_report_sha256=before,submission_sha256=digest(path))\\n    (OUT/\'run_manifest.json\').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\\n    print(json.dumps({k:report[k] for k in [\'stage\',\'rows_verified\',\'test_changes_vs_best\',\'dev_delta\',\'original_cv_gate_passed\',\'additional_dev_gates_passed\',\'candidate_sha256\']}),flush=True)\\n\\nif __name__==\'__main__\':main()\\n"}')
for name,source in modules51.items():
    (SRC43/name).write_text(source)
parent51=Path('/content/ds102_v50/run');parent51.mkdir(parents=True,exist_ok=True)
(parent51/'metrics_v50.json').write_text('{\n  "config": {\n    "rule": "Keep original class1 predictions; change0/2/3 only when both independent feature-view predictions agree on0/2/3",\n    "catalogue_alpha": 0.3,\n    "supervised_alpha": 0.15,\n    "frozen_classes": [\n      1\n    ],\n    "seed": 102,\n    "min_oof_gain": 0.002,\n    "min_positive_folds": 2,\n    "max_fold_drop": 0.003,\n    "min_dev_gain": 0.001,\n    "require_bootstrap_q05_positive": true,\n    "baseline_user_reported_kaggle": 0.77436\n  },\n  "pretrained": false,\n  "external_data": false,\n  "execution": "hosted Colab CPU",\n  "fit_data": "train only; fixed vote rule, no trained meta-parameters",\n  "submission_created": false,\n  "selection_caveat": "Rule proposed after prior model and two partial V49 committee fold outcomes; repeated OOF/dev selection remains. Paired bootstrap is conditional, not post-selection proof or guaranteed Kaggle gain.",\n  "stage": "rejected_on_cv",\n  "elapsed_seconds": 20.499454975128174,\n  "active_fold": null,\n  "fold_deltas": [\n    0.002593627858798242,\n    -0.00011642235028075376,\n    0.0018958846293559706\n  ],\n  "delta": 0.001463981731077979,\n  "baseline_oof": {\n    "macro_f1": 0.7749451430616494,\n    "accuracy": 0.8978051167342374,\n    "classification_report": {\n      "0": {\n        "precision": 0.9235403151065802,\n        "recall": 0.9479642313546424,\n        "f1-score": 0.9355929020749225,\n        "support": 10512.0\n      },\n      "1": {\n        "precision": 0.6908212560386473,\n        "recall": 0.6777251184834123,\n        "f1-score": 0.6842105263157895,\n        "support": 211.0\n      },\n      "2": {\n        "precision": 0.628968253968254,\n        "recall": 0.5747960108794198,\n        "f1-score": 0.600663192799621,\n        "support": 1103.0\n      },\n      "3": {\n        "precision": 0.913515862668405,\n        "recall": 0.8475806451612903,\n        "f1-score": 0.8793139510562644,\n        "support": 2480.0\n      },\n      "accuracy": 0.8978051167342374,\n      "macro avg": {\n        "precision": 0.7892114219454717,\n        "recall": 0.7620165014696912,\n        "f1-score": 0.7749451430616494,\n        "support": 14306.0\n      },\n      "weighted avg": {\n        "precision": 0.8956584930077698,\n        "recall": 0.8978051167342374,\n        "f1-score": 0.8963058232868542,\n        "support": 14306.0\n      }\n    },\n    "confusion_matrix": [\n      [\n        9965,\n        47,\n        338,\n        162\n      ],\n      [\n        54,\n        143,\n        3,\n        11\n      ],\n      [\n        443,\n        0,\n        634,\n        26\n      ],\n      [\n        328,\n        17,\n        33,\n        2102\n      ]\n    ]\n  },\n  "candidate_oof": {\n    "macro_f1": 0.7764091247927274,\n    "accuracy": 0.8987138263665595,\n    "classification_report": {\n      "0": {\n        "precision": 0.924167979975897,\n        "recall": 0.9483447488584474,\n        "f1-score": 0.9361002863984225,\n        "support": 10512.0\n      },\n      "1": {\n        "precision": 0.6908212560386473,\n        "recall": 0.6777251184834123,\n        "f1-score": 0.6842105263157895,\n        "support": 211.0\n      },\n      "2": {\n        "precision": 0.6307996051332675,\n        "recall": 0.5793291024478694,\n        "f1-score": 0.6039697542533081,\n        "support": 1103.0\n      },\n      "3": {\n        "precision": 0.9160504567203132,\n        "recall": 0.8491935483870968,\n        "f1-score": 0.8813559322033898,\n        "support": 2480.0\n      },\n      "accuracy": 0.8987138263665595,\n      "macro avg": {\n        "precision": 0.7904598244670313,\n        "recall": 0.7636481295442065,\n        "f1-score": 0.7764091247927274,\n        "support": 14306.0\n      },\n      "weighted avg": {\n        "precision": 0.8967002787403296,\n        "recall": 0.8987138263665595,\n        "f1-score": 0.8972875704235045,\n        "support": 14306.0\n      }\n    },\n    "confusion_matrix": [\n      [\n        9969,\n        47,\n        338,\n        158\n      ],\n      [\n        54,\n        143,\n        3,\n        11\n      ],\n      [\n        440,\n        0,\n        639,\n        24\n      ],\n      [\n        324,\n        17,\n        33,\n        2106\n      ]\n    ]\n  },\n  "novel_fake_f1_delta": 0.0,\n  "class1_pred_mask_exactly_preserved": true,\n  "bootstrap": {\n    "observed_delta": 0.001463981731077979,\n    "mean": 0.0014749578989163286,\n    "sd": 0.0006669067118048589,\n    "q025": 0.00016230520677764927,\n    "q05": 0.00040480283987543445,\n    "q95": 0.002604152109641072,\n    "q975": 0.002795120536474266,\n    "fraction_delta_gt_0": 0.989,\n    "groups": 13656,\n    "rows": 14306,\n    "resamples": 2000,\n    "seed": 102,\n    "fraction_resamples_missing_a_class": 0.0,\n    "intervals": "q025-q975: central 95%; q05-q95: central 90%",\n    "caveat": "Conditional on these selected/fitted predictions. Does not remove train/validation template overlap, selection bias, or training-sample uncertainty. Positive fraction is not a posterior probability or a calibrated p-value."\n  },\n  "gates": {\n    "oof": false,\n    "folds": true,\n    "worst_fold": true,\n    "novel_fake": true,\n    "bootstrap": true\n  }\n}\n')
(parent51/'run_manifest.json').write_text('{\n  "config": {\n    "rule": "Keep original class1 predictions; change0/2/3 only when both independent feature-view predictions agree on0/2/3",\n    "catalogue_alpha": 0.3,\n    "supervised_alpha": 0.15,\n    "frozen_classes": [\n      1\n    ],\n    "seed": 102,\n    "min_oof_gain": 0.002,\n    "min_positive_folds": 2,\n    "max_fold_drop": 0.003,\n    "min_dev_gain": 0.001,\n    "require_bootstrap_q05_positive": true,\n    "baseline_user_reported_kaggle": 0.77436\n  },\n  "versions": {\n    "numpy": "2.4.6",\n    "scipy": "1.16.3",\n    "pandas": "2.2.3",\n    "scikit-learn": "1.7.2",\n    "catboost": "1.2.10"\n  },\n  "inputs": {\n    "data/ds102_train.csv": "15202b19b2dc2c6c89afeb209024bb0dc77ddf927de44381a96c24a961d823a5",\n    "data/ds102_dev.csv": "d895cc298b1e5b56bbb95c9f9b15610e5207a00cfd12cc79f32061e27b7a8ccf",\n    "data/ds102_test.csv": "82c21afd6f8f4610912bc12ed19c3b6b1a586fb0dfa5d2296f6548ae4fc6b398",\n    "data/ds102_samplesubmission.csv": "4cf7832636057152395b8b20302c11a691fd2ab35d50057ef43443f3d388424d",\n    "baseline/reference_v3.npz": "a533936e9004392f329d9d201d3ab93d2ef97470bbbea3d027da8ad477ed513c",\n    "baseline/latent_oof.npz": "1bb83294b662750812e8f633acb52ced4d554d5ba1f9989f33d4cccb3c52aeab",\n    "baseline/latent_oof_manifest.json": "bc8efe4355565b4ca26880a212f86e82feeafd874c8e39f221575f57d09ba5c7",\n    "baseline/latent_full.npz": "12c9550710236debfb8e1aed472a1180d19fe376629a9512ec609f253548b2d4",\n    "baseline/latent_full_manifest.json": "3205956fe22cd6289cc339f785a906a49b0b40348d6c1ecc695569df9c5fb74a",\n    "baseline/submission_best_v11.csv": "05bffeff984bd4927c8dd3e9bd3613aef6a7abce303e6e1dcb7cfa4a12012059"\n  },\n  "parent_manifest_sha256": {\n    "47": "3c80a99769510e2554514e65b3eb0dd436f0e203074568551a6820ed4c1cb3f7",\n    "48": "9b196fbc495ac7bb7cdc94e7b946e9a0f8e01d62ecc0ae01d3c9440be6d8ca2e"\n  },\n  "prior49_manifest_sha256": "2365d1c868844cb1c01c9624a1c6715dba0eb2b446740b5c5b3cf30fd144b372",\n  "sources": {\n    "select_v2.py": "c4ae5f81f5993ffb00a69d4142c302caa7f2c7692a2784c1469419be46ecb264",\n    "supervised_svd_v48.py": "8f194156ec49b40fdc92fab2f7f634d3476bf0eb94d9c6d8dc6b99dea1880bff",\n    "experiment_v50.py": "75ae784634b37680b7a5d08c146dfb6ecf474d9738fd3a80410bb55bf75d865e",\n    "latent_v6.py": "57a5899f26f582d192e5963c7d16074a5a4b117b725eddb290135f19c70cc489",\n    "handoff_v43.py": "78e009e54727ec319c9b9c34c5afc29585cd021dddc1a737d217cc8964b8d332",\n    "review_common.py": "1c2570952831186dab48d977afed3ee7099f4b82c3c2f68aff08d58a69e3d2c4",\n    "codex_checked_common.py": "7eb8983f88d924ee247e5459f8c9a7ba103d27f80c37de73e88d64c116ae0bff",\n    "train_submit_v2.py": "36d9d2888bf3572ba053f8ba20e5acd20585c341529bc94944e384ad9108d5a3",\n    "lexical_chi2_v41.py": "e3b549d7d967ffd1ca389bd0373a09d7c0d80a4bad1fc9753db85602f6b3deb3",\n    "experiment_v49.py": "31d561ae2155c3b1d7cac4d7d2fc449876a9285e5ebb69f56cd70dbda9aff7c3",\n    "improved_features.py": "7229403b55d75a17d4376142c3f867034fbba368da3548ac857335a546d242b5",\n    "catboost_v3.py": "1f13474be5eff7ab48633f3ba106b7634c802a95e2b8da6ef2ba3b026ccafbf1",\n    "train_submit.py": "895d2d0c3c2649b990046859ba811205b25d84cbf935a9ad506573d5cddd901a",\n    "experiment_v6.py": "694e93dc12aacde93ffe115ea9c3c01effe54e08a4bd39ca43a3c808ea5bfc54",\n    "train_submit_v6.py": "e9a319cfe41f6b73012545461ab0bab70e0fd6228b456b353eab88518c01adcd",\n    "experiment_v48.py": "4eee05dd6fede544296a575ee73dd80b3df4db6ff545ff972e4622ea3c204eb1",\n    "selection_v5.py": "758bc7eee40c987d6bf63f65bd2040324a74dbea1d4ab60e78d5cf24a66c9752",\n    "catalogue_svd_v40.py": "116be1290193f469977bb6bd71dc00b3206134e975f387a3ec33e83e1474893a",\n    "cloud_checks.py": "d61a20a7f8c32f8a1dbdf227ec417adbbcb63a1eb45323cbeffa8ae6fccb0f21"\n  }\n}\n')
print('Đã ghi mã nguồn và bản sao bằng chứng CV; không có nhãn test.')


## 4. Cấu hình lượt thử

In [ ]:
OUT43=Path('/content/ds102_v52_exploratory');OUT43.mkdir(parents=True,exist_ok=True)
print('V52: fit lại hai nhánh từ train; báo cáo rõ các gate gốc không đạt và xuất CSV thử nghiệm khi các mức điểm quan sát đều tăng.')


## 5. Chạy và theo dõi

In [ ]:
import subprocess,os,zipfile
command43=[str(PY43),'-u',str(SRC43/'export_v52_exploratory.py')]
process43=subprocess.Popen(command43,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
    cwd=SRC43,env={**os.environ,'OMP_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2','MKL_NUM_THREADS':'2'})
try:
    with (OUT43/'training.log').open('a') as log43:
        for line43 in process43.stdout:
            print(line43,end='',flush=True);log43.write(line43);log43.flush()
    exit43=process43.wait()
    print('\nKết thúc, mã trạng thái:',exit43)
except KeyboardInterrupt:
    process43.terminate()
    try:process43.wait(timeout=20)
    except subprocess.TimeoutExpired:process43.kill();process43.wait()
    print('Đã dừng lượt chạy. Checkpoint các fold hoàn tất được giữ trong runtime này.')
finally:
    FEEDBACK43=OUT43/'feedback_v52.zip'
    with zipfile.ZipFile(FEEDBACK43,'w',zipfile.ZIP_DEFLATED) as feedback43:
        for name43 in ['metrics_v52_exploratory.json','run_manifest.json','environment.json','training.log','error.txt']:
            path43=OUT43/name43
            if path43.exists():feedback43.write(path43,name43)
    print('Báo cáo để gửi lại:',FEEDBACK43)


## 6. Tải báo cáo gửi lại

In [ ]:
import json,zipfile
from google.colab import files
FEEDBACK43=OUT43/'feedback_v52.zip'
with zipfile.ZipFile(FEEDBACK43,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['metrics_v52_exploratory.json','run_manifest.json','training.log','error.txt']:
        p=OUT43/name
        if p.exists():z.write(p,name)
report43=OUT43/'metrics_v52_exploratory.json'
summary43=json.loads(report43.read_text()) if report43.exists() else {}
print('Trạng thái:',summary43.get('stage'))
print('Chênh lệch dev:',summary43.get('dev_delta'))
print('Gate CV gốc đạt:',summary43.get('original_cv_gate_passed'))
print('Gate dev bổ sung đạt:',summary43.get('additional_dev_gates_passed'))
files.download(str(FEEDBACK43))
candidate43=OUT43/'submission_v52_exploratory.csv'
if candidate43.exists() and summary43.get('stage')=='exploratory_candidate_ready_not_submitted':
    print('CSV thử nghiệm: chưa qua các gate ngặt ban đầu và chưa biết điểm Kaggle. Chỉ nộp CSV nếu bạn muốn kiểm chứng.')
    files.download(str(candidate43))
else:
    print('Chưa có CSV. Gửi feedback_v52.zip để kiểm tra log.')
